In [25]:
from pathlib import Path
import pandas as pd
import numpy as np
import re
import openpyxl
import win32com.client as win32
import shutil

PROJECT_ROOT = Path(
    r"D:\My Drive\BOP_OCTC_2025"
)

XLS_DIR = (
    PROJECT_ROOT
    / "Photos"
    / "2026"
    / "Cropped"
    / "XLS_OutputFiles"
)

REDO_DIR = (
    PROJECT_ROOT
    / "Photos"
    / "2026"
    / "Cropped"
    / "Incomplete_Photos"
    / "Incomplete_XLS"
)

MASTER_SURVEY = (
    PROJECT_ROOT
    / "2026 Master Survey.xlsx"
)

# This is the master containing the blank placeholder rows
OLD_RECOVERED_MASTER = (
    XLS_DIR
    / "2026_SamplePoint_Camera_Masters_RECOVERED.xlsx"
)

# Write a NEW file. Do not overwrite the old recovery product yet.
PATCHED_MASTER = (
    XLS_DIR
    / "2026_SamplePoint_Camera_Masters_RECOVERED_COMPLETE.xlsx"
)

print("Redo directory:")
print(REDO_DIR)

print("\nOld recovered master:")
print(OLD_RECOVERED_MASTER)

print("\nNew patched master:")
print(PATCHED_MASTER)

assert REDO_DIR.exists()
assert MASTER_SURVEY.exists()
assert OLD_RECOVERED_MASTER.exists()

Redo directory:
D:\My Drive\BOP_OCTC_2025\Photos\2026\Cropped\Incomplete_Photos\Incomplete_XLS

Old recovered master:
D:\My Drive\BOP_OCTC_2025\Photos\2026\Cropped\XLS_OutputFiles\2026_SamplePoint_Camera_Masters_RECOVERED.xlsx

New patched master:
D:\My Drive\BOP_OCTC_2025\Photos\2026\Cropped\XLS_OutputFiles\2026_SamplePoint_Camera_Masters_RECOVERED_COMPLETE.xlsx


In [26]:
REDO_FILES = sorted(
    [
        p for p in REDO_DIR.iterdir()
        if p.is_file()
        and p.suffix.lower() == ".xls"
        and (
            p.stem.upper().startswith("AW120_INCOMPLETE")
            or p.stem.upper().startswith("CAM5_INCOMPLETE")
        )
    ],
    key=lambda p: p.name.lower()
)

print(f"Redo XLS files found: {len(REDO_FILES)}\n")

for p in REDO_FILES:
    print(" ", p.name)

if len(REDO_FILES) != 6:
    raise ValueError(
        f"Expected 6 redo XLS files, found {len(REDO_FILES)}."
    )

Redo XLS files found: 6

  AW120_INCOMPLETE.XLS
  AW120_INCOMPLETE_2.XLS
  CAM5_INCOMPLETE_1.XLS
  CAM5_INCOMPLETE_2.XLS
  CAM5_INCOMPLETE_3.XLS
  CAM5_INCOMPLETE_4.XLS


In [27]:
def normalize_header(value):
    if value is None:
        return ""

    return re.sub(
        r"[^a-z0-9]+",
        "",
        str(value).strip().lower()
    )


def has_value(value):
    """
    True only for genuinely populated cells.

    Correctly treats:
        None
        NaN
        pd.NA
        ""
        whitespace-only strings

    as missing.
    """

    if value is None:
        return False

    try:
        if pd.isna(value):
            return False
    except (TypeError, ValueError):
        pass

    return str(value).strip() != ""

def extract_photo_number(value):
    """
    Handles:
        DSCN9173_c.jpg
        DSCN0001_c.jpg
        9173
        9173.0
    """

    if value is None:
        return None

    if isinstance(value, (int, np.integer)):
        return int(value)

    if isinstance(value, (float, np.floating)):
        if np.isnan(value):
            return None

        if float(value).is_integer():
            return int(value)

    text = str(value).strip()

    match = re.search(
        r"DSCN0*(\d+)",
        text,
        flags=re.IGNORECASE
    )

    if match:
        return int(match.group(1))

    match = re.fullmatch(
        r"0*(\d+)(?:\.0+)?",
        text
    )

    if match:
        return int(match.group(1))

    return None


def camera_from_redo_filename(path):
    name = path.stem.upper()

    if name.startswith("AW120_INCOMPLETE"):
        return "AW120"

    if name.startswith("CAM5_INCOMPLETE"):
        return "Cam5"

    return None

In [28]:
def read_redo_xls(excel, path):

    wb = None

    try:
        wb = excel.Workbooks.Open(
            Filename=str(path),
            ReadOnly=True,
            AddToMru=False,
        )

        ws = wb.Worksheets(1)

        values = ws.UsedRange.Value

        if values is None:
            raise ValueError(
                f"No data found in {path.name}"
            )

        # Force 2-D
        if not isinstance(values, tuple):
            values = ((values,),)

        elif len(values) and not isinstance(values[0], tuple):
            values = (values,)

        header_row_idx = None
        header = None

        for r_i, row in enumerate(values[:30]):

            norm = [
                normalize_header(v)
                for v in row
            ]

            if (
                "point1" in norm
                and "point100" in norm
                and (
                    "image" in norm
                    or "photo" in norm
                    or "photos" in norm
                )
            ):
                header_row_idx = r_i
                header = list(row)
                break

        if header_row_idx is None:
            raise ValueError(
                f"Could not locate SamplePoint header "
                f"in {path.name}"
            )

        hmap = {
            normalize_header(v): i
            for i, v in enumerate(header)
            if v is not None
        }

        image_idx = next(
            hmap[x]
            for x in ["image", "photo", "photos"]
            if x in hmap
        )

        rows = []

        for excel_row, raw_row in enumerate(
            values[header_row_idx + 1:],
            start=header_row_idx + 2
        ):

            row = list(raw_row)

            # pad short rows
            if len(row) < len(header):
                row.extend(
                    [None] * (len(header) - len(row))
                )

            image = row[image_idx]
            photo_number = extract_photo_number(image)

            if photo_number is None:
                continue

            rows.append({
                "ExcelRow": excel_row,
                "Image": image,
                "PhotoNumber": photo_number,
                "_row": row,
                "_hmap": hmap,
            })

        return rows

    finally:

        if wb is not None:
            wb.Close(SaveChanges=False)

In [29]:
redo_records = []

excel = win32.DispatchEx(
    "Excel.Application"
)

excel.Visible = False
excel.DisplayAlerts = False
excel.ScreenUpdating = False

try:

    for path in REDO_FILES:

        camera = camera_from_redo_filename(path)

        rows = read_redo_xls(
            excel,
            path
        )

        print(
            f"{path.name:28s} "
            f"camera={camera:5s} "
            f"rows={len(rows):4d}"
        )

        for rec in rows:

            rec["CameraAssignment"] = camera
            rec["RedoFile"] = path.name

            point1_idx = rec["_hmap"]["point1"]

            rec["Point1Present"] = has_value(
                rec["_row"][point1_idx]
            )

            redo_records.append(rec)

finally:

    excel.Quit()


redo_df = pd.DataFrame(
    redo_records
)

print("\n" + "=" * 70)
print("REDO INVENTORY")
print("=" * 70)

print(
    f"Rows with photo IDs: "
    f"{len(redo_df):,}"
)

print(
    f"Rows with Point1:    "
    f"{redo_df['Point1Present'].sum():,}"
)

display(
    redo_df.groupby(
        [
            "CameraAssignment",
            "RedoFile"
        ],
        as_index=False
    ).agg(
        Photos=("PhotoNumber", "size"),
        UniquePhotos=("PhotoNumber", "nunique"),
        Completed=("Point1Present", "sum"),
    )
)

AW120_INCOMPLETE.XLS         camera=AW120 rows= 138
AW120_INCOMPLETE_2.XLS       camera=AW120 rows=  30
CAM5_INCOMPLETE_1.XLS        camera=Cam5  rows=  65
CAM5_INCOMPLETE_2.XLS        camera=Cam5  rows=  65
CAM5_INCOMPLETE_3.XLS        camera=Cam5  rows=  70
CAM5_INCOMPLETE_4.XLS        camera=Cam5  rows= 100

REDO INVENTORY
Rows with photo IDs: 468
Rows with Point1:    378


,CameraAssignment,RedoFile,Photos,UniquePhotos,Completed
0,AW120,AW120_INCOMPLETE.XLS,138,138,48
1,AW120,AW120_INCOMPLETE_2.XLS,30,30,30
2,Cam5,CAM5_INCOMPLETE_1.XLS,65,65,65
3,Cam5,CAM5_INCOMPLETE_2.XLS,65,65,65
4,Cam5,CAM5_INCOMPLETE_3.XLS,70,70,70
5,Cam5,CAM5_INCOMPLETE_4.XLS,100,100,100


In [30]:
completed_redo = (
    redo_df[
        redo_df["Point1Present"]
    ]
    .copy()
)

redo_duplicates = (
    completed_redo[
        completed_redo.duplicated(
            subset=[
                "CameraAssignment",
                "PhotoNumber",
            ],
            keep=False
        )
    ]
    .sort_values(
        [
            "CameraAssignment",
            "PhotoNumber",
            "RedoFile",
        ]
    )
)

print(
    f"Completed redo rows: "
    f"{len(completed_redo):,}"
)

print(
    f"Unique completed camera × photo keys: "
    f"{completed_redo[
        ['CameraAssignment', 'PhotoNumber']
    ].drop_duplicates().shape[0]:,}"
)

print(
    f"Duplicate completed keys: "
    f"{redo_duplicates[
        ['CameraAssignment', 'PhotoNumber']
    ].drop_duplicates().shape[0]:,}"
)

display(
    redo_duplicates[
        [
            "CameraAssignment",
            "PhotoNumber",
            "RedoFile",
            "ExcelRow",
            "Image",
        ]
    ]
)

Completed redo rows: 378
Unique completed camera × photo keys: 378
Duplicate completed keys: 0


,CameraAssignment,PhotoNumber,RedoFile,ExcelRow,Image


In [31]:
# ============================================================
# IDENTIFY BLANK ROWS IN THE OLD RECOVERED MASTER
# Robust to Image / Photo / Photos column names
# ============================================================

def find_df_column(df, candidates):
    """
    Find a dataframe column by normalized name.
    """

    lookup = {
        normalize_header(col): col
        for col in df.columns
    }

    for candidate in candidates:

        candidate_norm = normalize_header(
            candidate
        )

        if candidate_norm in lookup:
            return lookup[candidate_norm]

    raise KeyError(
        f"Could not find any of {candidates}.\n"
        f"Available columns:\n{list(df.columns)}"
    )


master_frames = []

for sheet_name, camera in [
    ("AW120_Master", "AW120"),
    ("Cam5_Master", "Cam5"),
]:

    df = pd.read_excel(
        OLD_RECOVERED_MASTER,
        sheet_name=sheet_name
    )

    # --------------------------------------------------------
    # Find actual photo/image column
    # --------------------------------------------------------

    photo_col = find_df_column(
        df,
        [
            "Image",
            "Photo",
            "Photos",
        ]
    )

    print(
        f"{sheet_name}: "
        f"photo field = {photo_col!r}"
    )

    df["CameraAssignment"] = camera
    df["MasterSheet"] = sheet_name

    # Excel header is row 1
    df["ExcelRow"] = np.arange(
        2,
        len(df) + 2
    )

    df["PhotoNumber"] = (
        df[photo_col]
        .map(extract_photo_number)
        .astype("Int64")
    )

    # --------------------------------------------------------
    # Locate Point1-Point100 robustly
    # --------------------------------------------------------

    point_cols = []

    for n in range(1, 101):

        col = find_df_column(
            df,
            [f"Point{n}"]
        )

        point_cols.append(col)

    # --------------------------------------------------------
    # Classification completeness
    # --------------------------------------------------------

    df["All100Blank"] = (
        df[point_cols]
        .apply(
            lambda row: all(
                not has_value(v)
                for v in row
            ),
            axis=1
        )
    )

    df["All100Present"] = (
        df[point_cols]
        .apply(
            lambda row: all(
                has_value(v)
                for v in row
            ),
            axis=1
        )
    )

    df["PartialClassification"] = (
        ~df["All100Blank"]
        & ~df["All100Present"]
    )

    master_frames.append(df)


old_master = pd.concat(
    master_frames,
    ignore_index=True
)

blank_master = (
    old_master[
        old_master["All100Blank"]
    ]
    .copy()
)


print("\n" + "=" * 72)
print("OLD RECOVERED MASTER STATUS")
print("=" * 72)

print(
    f"Total photo rows:        "
    f"{len(old_master):,}"
)

print(
    f"All 100 populated:       "
    f"{old_master['All100Present'].sum():,}"
)

print(
    f"All 100 blank:           "
    f"{old_master['All100Blank'].sum():,}"
)

print(
    f"Partial classifications: "
    f"{old_master['PartialClassification'].sum():,}"
)

display(
    blank_master[
        [
            "CameraAssignment",
            "PhotoNumber",
            "MasterSheet",
            "ExcelRow",
        ]
    ]
    .sort_values(
        [
            "CameraAssignment",
            "PhotoNumber",
        ]
    )
)

AW120_Master: photo field = 'image'
Cam5_Master: photo field = 'image'

OLD RECOVERED MASTER STATUS
Total photo rows:        2,280
All 100 populated:       1,897
All 100 blank:           383
Partial classifications: 0


,CameraAssignment,PhotoNumber,MasterSheet,ExcelRow
1150,AW120,9028,AW120_Master,1152
1151,AW120,9029,AW120_Master,1153
1152,AW120,9030,AW120_Master,1154
1153,AW120,9031,AW120_Master,1155
1154,AW120,9032,AW120_Master,1156
...,...,...,...,...
1585,Cam5,8785,Cam5_Master,322
1586,Cam5,8786,Cam5_Master,323
1587,Cam5,8787,Cam5_Master,324
1588,Cam5,8788,Cam5_Master,325


In [32]:
blank_keys = (
    blank_master[
        [
            "CameraAssignment",
            "PhotoNumber",
        ]
    ]
    .drop_duplicates()
)

redo_keys = (
    completed_redo[
        [
            "CameraAssignment",
            "PhotoNumber",
            "RedoFile",
            "ExcelRow",
        ]
    ]
    .drop_duplicates()
)

coverage = blank_keys.merge(
    redo_keys,
    on=[
        "CameraAssignment",
        "PhotoNumber",
    ],
    how="left",
    indicator=True
)

coverage["RedoFound"] = (
    coverage["_merge"] == "both"
)

print("=" * 72)
print("BLANK MASTER ↔ REDO COVERAGE")
print("=" * 72)

print(
    f"Blank camera/photo keys: "
    f"{len(coverage):,}"
)

print(
    f"Recovered by redo XLS:   "
    f"{coverage['RedoFound'].sum():,}"
)

print(
    f"Still not recovered:     "
    f"{(~coverage['RedoFound']).sum():,}"
)

display(
    coverage[
        ~coverage["RedoFound"]
    ]
)

BLANK MASTER ↔ REDO COVERAGE
Blank camera/photo keys: 383
Recovered by redo XLS:   378
Still not recovered:     5


,CameraAssignment,PhotoNumber,RedoFile,ExcelRow,_merge,RedoFound
0,AW120,9435,NaN,NaN,left_only,False
1,AW120,9436,NaN,NaN,left_only,False
2,AW120,9437,NaN,NaN,left_only,False
3,AW120,9438,NaN,NaN,left_only,False
4,AW120,9439,NaN,NaN,left_only,False


In [33]:
# ============================================================
# IDENTIFY BLANK ROWS IN THE OLD RECOVERED MASTER
# Robust to Image / Photo / Photos column names
# ============================================================

def find_df_column(df, candidates):
    """
    Find a dataframe column by normalized name.
    """

    lookup = {
        normalize_header(col): col
        for col in df.columns
    }

    for candidate in candidates:

        candidate_norm = normalize_header(
            candidate
        )

        if candidate_norm in lookup:
            return lookup[candidate_norm]

    raise KeyError(
        f"Could not find any of {candidates}.\n"
        f"Available columns:\n{list(df.columns)}"
    )


master_frames = []

for sheet_name, camera in [
    ("AW120_Master", "AW120"),
    ("Cam5_Master", "Cam5"),
]:

    df = pd.read_excel(
        OLD_RECOVERED_MASTER,
        sheet_name=sheet_name
    )

    # --------------------------------------------------------
    # Find actual photo/image column
    # --------------------------------------------------------

    photo_col = find_df_column(
        df,
        [
            "Image",
            "Photo",
            "Photos",
        ]
    )

    print(
        f"{sheet_name}: "
        f"photo field = {photo_col!r}"
    )

    df["CameraAssignment"] = camera
    df["MasterSheet"] = sheet_name

    # Excel header is row 1
    df["ExcelRow"] = np.arange(
        2,
        len(df) + 2
    )

    df["PhotoNumber"] = (
        df[photo_col]
        .map(extract_photo_number)
        .astype("Int64")
    )

    # --------------------------------------------------------
    # Locate Point1-Point100 robustly
    # --------------------------------------------------------

    point_cols = []

    for n in range(1, 101):

        col = find_df_column(
            df,
            [f"Point{n}"]
        )

        point_cols.append(col)

    # --------------------------------------------------------
    # Classification completeness
    # --------------------------------------------------------

    df["All100Blank"] = (
        df[point_cols]
        .apply(
            lambda row: all(
                not has_value(v)
                for v in row
            ),
            axis=1
        )
    )

    df["All100Present"] = (
        df[point_cols]
        .apply(
            lambda row: all(
                has_value(v)
                for v in row
            ),
            axis=1
        )
    )

    df["PartialClassification"] = (
        ~df["All100Blank"]
        & ~df["All100Present"]
    )

    master_frames.append(df)


old_master = pd.concat(
    master_frames,
    ignore_index=True
)

blank_master = (
    old_master[
        old_master["All100Blank"]
    ]
    .copy()
)


print("\n" + "=" * 72)
print("OLD RECOVERED MASTER STATUS")
print("=" * 72)

print(
    f"Total photo rows:        "
    f"{len(old_master):,}"
)

print(
    f"All 100 populated:       "
    f"{old_master['All100Present'].sum():,}"
)

print(
    f"All 100 blank:           "
    f"{old_master['All100Blank'].sum():,}"
)

print(
    f"Partial classifications: "
    f"{old_master['PartialClassification'].sum():,}"
)

display(
    blank_master[
        [
            "CameraAssignment",
            "PhotoNumber",
            "MasterSheet",
            "ExcelRow",
        ]
    ]
    .sort_values(
        [
            "CameraAssignment",
            "PhotoNumber",
        ]
    )
)

AW120_Master: photo field = 'image'
Cam5_Master: photo field = 'image'

OLD RECOVERED MASTER STATUS
Total photo rows:        2,280
All 100 populated:       1,897
All 100 blank:           383
Partial classifications: 0


,CameraAssignment,PhotoNumber,MasterSheet,ExcelRow
1150,AW120,9028,AW120_Master,1152
1151,AW120,9029,AW120_Master,1153
1152,AW120,9030,AW120_Master,1154
1153,AW120,9031,AW120_Master,1155
1154,AW120,9032,AW120_Master,1156
...,...,...,...,...
1585,Cam5,8785,Cam5_Master,322
1586,Cam5,8786,Cam5_Master,323
1587,Cam5,8787,Cam5_Master,324
1588,Cam5,8788,Cam5_Master,325


In [34]:
blank_key_set = set(
    zip(
        blank_master["CameraAssignment"],
        blank_master["PhotoNumber"].astype(int)
    )
)

redo_key_set = set(
    zip(
        completed_redo["CameraAssignment"],
        completed_redo["PhotoNumber"].astype(int)
    )
)

unexpected_redos = sorted(
    redo_key_set - blank_key_set
)

missing_redos = sorted(
    blank_key_set - redo_key_set
)

print(
    f"Redo keys not corresponding "
    f"to a blank master row: "
    f"{len(unexpected_redos):,}"
)

print(
    f"Blank master keys lacking redo: "
    f"{len(missing_redos):,}"
)

if unexpected_redos:
    print("\nUnexpected redo keys:")
    display(
        pd.DataFrame(
            unexpected_redos,
            columns=[
                "CameraAssignment",
                "PhotoNumber"
            ]
        )
    )

if missing_redos:
    print("\nStill-missing keys:")
    display(
        pd.DataFrame(
            missing_redos,
            columns=[
                "CameraAssignment",
                "PhotoNumber"
            ]
        )
    )

Redo keys not corresponding to a blank master row: 0
Blank master keys lacking redo: 5

Still-missing keys:


,CameraAssignment,PhotoNumber
0,AW120,9435
1,AW120,9436
2,AW120,9437
3,AW120,9438
4,AW120,9439


In [35]:
# ============================================================
# CURRENT ORIGINAL DATABASES THAT WERE REPAIRED IN PLACE
# ============================================================

DIRECTLY_RECOVERED_FILES = [
    XLS_DIR / "GARRETT_DATABASE_8.XLS",
    XLS_DIR / "CHLOE_DATABASE_5.XLS",
]

for path in DIRECTLY_RECOVERED_FILES:
    assert path.exists(), (
        f"Missing expected source database:\n{path}"
    )


direct_records = []

excel = win32.DispatchEx(
    "Excel.Application"
)

excel.Visible = False
excel.DisplayAlerts = False
excel.ScreenUpdating = False

try:

    for path in DIRECTLY_RECOVERED_FILES:

        rows = read_redo_xls(
            excel,
            path
        )

        for rec in rows:

            point1_idx = (
                rec["_hmap"]["point1"]
            )

            rec["Point1Present"] = (
                has_value(
                    rec["_row"][point1_idx]
                )
            )

            rec["SourceFile"] = path.name

            # Both of these databases are AW120
            rec["CameraAssignment"] = "AW120"

            direct_records.append(rec)

finally:

    excel.Quit()


direct_df = pd.DataFrame(
    direct_records
)


direct_summary = (
    direct_df
    .groupby(
        "SourceFile",
        as_index=False
    )
    .agg(
        Photos=(
            "PhotoNumber",
            "size"
        ),
        UniquePhotos=(
            "PhotoNumber",
            "nunique"
        ),
        Completed=(
            "Point1Present",
            "sum"
        ),
    )
)

display(direct_summary)

,SourceFile,Photos,UniquePhotos,Completed
0,CHLOE_DATABASE_5.XLS,80,80,75
1,GARRETT_DATABASE_8.XLS,80,80,80


In [36]:
# ============================================================
# COMPLETE RECOVERY CANDIDATE POOL
#
# 1. 378 classifications from INCOMPLETE_XLS
# 2. updated GARRETT_DATABASE_8
# 3. updated CHLOE_DATABASE_5
# ============================================================

redo_completed = (
    completed_redo[
        [
            "CameraAssignment",
            "PhotoNumber",
            "RedoFile",
            "ExcelRow",
            "_row",
            "_hmap",
        ]
    ]
    .copy()
)

redo_completed["RecoverySource"] = (
    "Incomplete_XLS"
)

redo_completed["SourceFile"] = (
    redo_completed["RedoFile"]
)


direct_completed = (
    direct_df[
        direct_df["Point1Present"]
    ][
        [
            "CameraAssignment",
            "PhotoNumber",
            "SourceFile",
            "ExcelRow",
            "_row",
            "_hmap",
        ]
    ]
    .copy()
)

direct_completed["RecoverySource"] = (
    "Updated original database"
)


recovery_candidates = pd.concat(
    [
        redo_completed[
            [
                "CameraAssignment",
                "PhotoNumber",
                "SourceFile",
                "ExcelRow",
                "_row",
                "_hmap",
                "RecoverySource",
            ]
        ],
        direct_completed[
            [
                "CameraAssignment",
                "PhotoNumber",
                "SourceFile",
                "ExcelRow",
                "_row",
                "_hmap",
                "RecoverySource",
            ]
        ],
    ],
    ignore_index=True
)


print(
    f"Total completed recovery candidate rows: "
    f"{len(recovery_candidates):,}"
)

print(
    f"Unique camera × photo keys: "
    f"{recovery_candidates[
        ['CameraAssignment', 'PhotoNumber']
    ].drop_duplicates().shape[0]:,}"
)

Total completed recovery candidate rows: 533
Unique camera × photo keys: 533


In [37]:
# ============================================================
# WHICH OF THE 443 BLANK MASTER ROWS CAN NOW BE FILLED?
# ============================================================

blank_keys = (
    blank_master[
        [
            "CameraAssignment",
            "PhotoNumber",
        ]
    ]
    .drop_duplicates()
    .copy()
)

candidate_keys = (
    recovery_candidates[
        [
            "CameraAssignment",
            "PhotoNumber",
        ]
    ]
    .drop_duplicates()
    .copy()
)

coverage = blank_keys.merge(
    candidate_keys.assign(
        RecoveryFound=True
    ),
    on=[
        "CameraAssignment",
        "PhotoNumber",
    ],
    how="left"
)

coverage["RecoveryFound"] = (
    coverage["RecoveryFound"]
    .fillna(False)
)

print("=" * 72)
print("BLANK MASTER ↔ AVAILABLE RECOVERY DATA")
print("=" * 72)

print(
    f"Blank master keys:     "
    f"{len(blank_keys):,}"
)

print(
    f"Recovery data found:   "
    f"{coverage['RecoveryFound'].sum():,}"
)

print(
    f"Still unresolved:      "
    f"{(~coverage['RecoveryFound']).sum():,}"
)

still_missing = (
    coverage[
        ~coverage["RecoveryFound"]
    ]
    .sort_values(
        [
            "CameraAssignment",
            "PhotoNumber",
        ]
    )
    .reset_index(drop=True)
)

print("\nStill unresolved:")
display(still_missing)

BLANK MASTER ↔ AVAILABLE RECOVERY DATA
Blank master keys:     383
Recovery data found:   383
Still unresolved:      0

Still unresolved:


,CameraAssignment,PhotoNumber,RecoveryFound


In [39]:
# ============================================================
# BUILD THE EXACT PATCH PLAN
#
# Target:
#   the 383 completely blank rows in the current recovered master
#
# Source:
#   the unique completed recovery candidate having the same
#   camera + photo number
# ============================================================

# Recovery candidate keys must be unique.
candidate_duplicate_keys = (
    recovery_candidates[
        recovery_candidates.duplicated(
            subset=[
                "CameraAssignment",
                "PhotoNumber",
            ],
            keep=False
        )
    ]
    .sort_values(
        [
            "CameraAssignment",
            "PhotoNumber",
            "SourceFile",
        ]
    )
)

print(
    f"Duplicate recovery candidate keys: "
    f"{len(candidate_duplicate_keys):,}"
)

if len(candidate_duplicate_keys):
    display(candidate_duplicate_keys)

    raise ValueError(
        "Recovery candidates are not unique by camera + photo."
    )


# ------------------------------------------------------------
# Keep target workbook location for every blank row
# ------------------------------------------------------------

blank_targets = (
    blank_master[
        [
            "CameraAssignment",
            "PhotoNumber",
            "MasterSheet",
            "ExcelRow",
        ]
    ]
    .copy()
)


# ------------------------------------------------------------
# Merge the blank target rows to recovery classifications
# ------------------------------------------------------------

patch_plan = (
    blank_targets
    .merge(
        recovery_candidates[
            [
                "CameraAssignment",
                "PhotoNumber",
                "SourceFile",
                "RecoverySource",
                "_row",
                "_hmap",
            ]
        ],
        on=[
            "CameraAssignment",
            "PhotoNumber",
        ],
        how="left",
        validate="one_to_one",
        indicator=True,
    )
)


print("=" * 72)
print("PATCH PLAN")
print("=" * 72)

print(
    f"Blank master rows:     "
    f"{len(blank_targets):,}"
)

print(
    f"Matched source rows:   "
    f"{(patch_plan['_merge'] == 'both').sum():,}"
)

print(
    f"Unmatched targets:     "
    f"{(patch_plan['_merge'] != 'both').sum():,}"
)


if (patch_plan["_merge"] != "both").any():

    display(
        patch_plan[
            patch_plan["_merge"] != "both"
        ]
    )

    raise ValueError(
        "One or more blank master rows lack a recovery source."
    )


# ------------------------------------------------------------
# Provenance summary
# ------------------------------------------------------------

print("\nRecovery source summary:")

display(
    patch_plan
    .groupby(
        [
            "RecoverySource",
            "SourceFile",
        ],
        as_index=False
    )
    .agg(
        Photos=(
            "PhotoNumber",
            "size"
        )
    )
    .sort_values(
        [
            "RecoverySource",
            "SourceFile",
        ]
    )
)

Duplicate recovery candidate keys: 0
PATCH PLAN
Blank master rows:     383
Matched source rows:   383
Unmatched targets:     0

Recovery source summary:


,RecoverySource,SourceFile,Photos
0,Incomplete_XLS,AW120_INCOMPLETE.XLS,48
1,Incomplete_XLS,AW120_INCOMPLETE_2.XLS,30
2,Incomplete_XLS,CAM5_INCOMPLETE_1.XLS,65
3,Incomplete_XLS,CAM5_INCOMPLETE_2.XLS,65
4,Incomplete_XLS,CAM5_INCOMPLETE_3.XLS,70
5,Incomplete_XLS,CAM5_INCOMPLETE_4.XLS,100
6,Updated original database,GARRETT_DATABASE_8.XLS,5


In [40]:
# ============================================================
# VERIFY SOURCE CLASSIFICATIONS ARE COMPLETE
# ============================================================

def source_has_all_100(row_values, hmap):

    for n in range(1, 101):

        field = normalize_header(
            f"Point{n}"
        )

        if field not in hmap:
            return False

        idx = hmap[field]

        if idx >= len(row_values):
            return False

        if not has_value(
            row_values[idx]
        ):
            return False

    return True


patch_plan["SourceAll100Present"] = [
    source_has_all_100(
        row_values,
        hmap
    )
    for row_values, hmap in zip(
        patch_plan["_row"],
        patch_plan["_hmap"],
    )
]


print(
    f"Patch sources with all 100 hits: "
    f"{patch_plan['SourceAll100Present'].sum():,} "
    f"/ {len(patch_plan):,}"
)


bad_sources = (
    patch_plan[
        ~patch_plan["SourceAll100Present"]
    ]
    [
        [
            "CameraAssignment",
            "PhotoNumber",
            "SourceFile",
            "RecoverySource",
        ]
    ]
)

display(bad_sources)


if len(bad_sources):
    raise ValueError(
        "At least one recovery source does not contain all 100 hits."
    )

Patch sources with all 100 hits: 383 / 383


,CameraAssignment,PhotoNumber,SourceFile,RecoverySource


In [41]:
# ============================================================
# WRITE COMPLETE RECOVERED MASTER
# ============================================================

PATCHED_MASTER = (
    XLS_DIR
    / "2026_SamplePoint_Camera_Masters_RECOVERED_COMPLETE.xlsx"
)

# Never edit the current recovered master in place.
shutil.copy2(
    OLD_RECOVERED_MASTER,
    PATCHED_MASTER
)


wb = openpyxl.load_workbook(
    PATCHED_MASTER
)

fields_to_patch = (
    ["comment"]
    + [
        f"point{n}"
        for n in range(1, 101)
    ]
)

patched = 0


for _, rec in patch_plan.iterrows():

    sheet_name = rec["MasterSheet"]
    excel_row = int(
        rec["ExcelRow"]
    )

    ws = wb[sheet_name]

    # Target workbook header map
    target_hmap = {
        normalize_header(cell.value): cell.column
        for cell in ws[1]
        if cell.value is not None
    }

    source_hmap = rec["_hmap"]
    source_row = rec["_row"]

    # --------------------------------------------------------
    # SAFETY GATE:
    # target must still have blank Point1
    # --------------------------------------------------------

    target_point1_col = (
        target_hmap["point1"]
    )

    current_point1 = ws.cell(
        row=excel_row,
        column=target_point1_col,
    ).value

    if has_value(current_point1):

        raise ValueError(
            f"Refusing to overwrite populated target: "
            f"{rec['CameraAssignment']} "
            f"{rec['PhotoNumber']} "
            f"{sheet_name} row {excel_row}"
        )

    # --------------------------------------------------------
    # Patch Comment + Point1-Point100 only
    # --------------------------------------------------------

    for field in fields_to_patch:

        if field not in target_hmap:
            raise KeyError(
                f"{field} missing from target "
                f"{sheet_name}"
            )

        if field not in source_hmap:
            raise KeyError(
                f"{field} missing from source "
                f"{rec['SourceFile']}"
            )

        target_col = (
            target_hmap[field]
        )

        source_idx = (
            source_hmap[field]
        )

        ws.cell(
            row=excel_row,
            column=target_col,
        ).value = (
            source_row[source_idx]
        )

    patched += 1


wb.save(
    PATCHED_MASTER
)

wb.close()


print("=" * 72)
print("PATCH COMPLETE")
print("=" * 72)

print(
    f"Rows patched: {patched:,}"
)

print(
    f"\nWritten:\n{PATCHED_MASTER}"
)

PATCH COMPLETE
Rows patched: 383

Written:
D:\My Drive\BOP_OCTC_2025\Photos\2026\Cropped\XLS_OutputFiles\2026_SamplePoint_Camera_Masters_RECOVERED_COMPLETE.xlsx


In [42]:
# ============================================================
# FINAL COMPLETENESS QA
# ============================================================

qa_rows = []

for sheet_name, camera in [
    ("AW120_Master", "AW120"),
    ("Cam5_Master", "Cam5"),
]:

    df = pd.read_excel(
        PATCHED_MASTER,
        sheet_name=sheet_name
    )

    point_cols = [
        find_df_column(
            df,
            [f"Point{n}"]
        )
        for n in range(1, 101)
    ]

    all_present = (
        df[point_cols]
        .apply(
            lambda row: all(
                has_value(v)
                for v in row
            ),
            axis=1
        )
    )

    all_blank = (
        df[point_cols]
        .apply(
            lambda row: all(
                not has_value(v)
                for v in row
            ),
            axis=1
        )
    )

    qa_rows.append({
        "Camera": camera,
        "Photos": len(df),
        "Complete100": int(
            all_present.sum()
        ),
        "Blank100": int(
            all_blank.sum()
        ),
        "Partial": int(
            (
                ~all_present
                & ~all_blank
            ).sum()
        ),
    })


final_qa = pd.DataFrame(
    qa_rows
)

display(final_qa)


print("\nTOTALS")

print(
    f"Photos:     "
    f"{final_qa['Photos'].sum():,}"
)

print(
    f"Complete:   "
    f"{final_qa['Complete100'].sum():,}"
)

print(
    f"Blank:      "
    f"{final_qa['Blank100'].sum():,}"
)

print(
    f"Partial:    "
    f"{final_qa['Partial'].sum():,}"
)


assert (
    final_qa["Photos"].sum()
    == 2280
)

assert (
    final_qa["Complete100"].sum()
    == 2280
)

assert (
    final_qa["Blank100"].sum()
    == 0
)

assert (
    final_qa["Partial"].sum()
    == 0
)

print(
    "\nPASS: all 2,280 photos contain "
    "Point1 through Point100."
)

,Camera,Photos,Complete100,Blank100,Partial
0,AW120,1265,1265,0,0
1,Cam5,1015,1015,0,0



TOTALS
Photos:     2,280
Complete:   2,280
Blank:      0
Partial:    0

PASS: all 2,280 photos contain Point1 through Point100.


In [43]:
# ============================================================
# FINAL PHOTO-IDENTITY QA
#
# Compare:
#   corrected 2026 Master Survey.xlsx
#       vs
#   RECOVERED_COMPLETE master
#
# Nothing is modified.
# ============================================================

COMPLETE_MASTER = PATCHED_MASTER

PHOTO_COLUMNS = [
    "Center Photoplot",
    "North Photoplot",
    "East Photoplot",
    "South Photoplot",
    "West Photoplot",
]


# ------------------------------------------------------------
# Build actual camera × photo keys from completed master
# ------------------------------------------------------------

master_key_frames = []

for sheet_name, camera in [
    ("AW120_Master", "AW120"),
    ("Cam5_Master", "Cam5"),
]:

    df = pd.read_excel(
        COMPLETE_MASTER,
        sheet_name=sheet_name
    )

    photo_col = find_df_column(
        df,
        ["Image", "Photo", "Photos"]
    )

    tmp = pd.DataFrame({
        "CameraAssignment": camera,
        "PhotoNumber": (
            df[photo_col]
            .map(extract_photo_number)
            .astype("Int64")
        ),
        "MasterSheet": sheet_name,
        "MasterExcelRow": np.arange(
            2,
            len(df) + 2
        ),
    })

    master_key_frames.append(tmp)


master_keys = pd.concat(
    master_key_frames,
    ignore_index=True
)

master_keys = master_keys[
    master_keys["PhotoNumber"].notna()
].copy()


# ------------------------------------------------------------
# Check master itself for duplicate camera × photo keys
# ------------------------------------------------------------

master_duplicates = (
    master_keys[
        master_keys.duplicated(
            ["CameraAssignment", "PhotoNumber"],
            keep=False
        )
    ]
    .sort_values(
        ["CameraAssignment", "PhotoNumber"]
    )
)

print(
    f"Duplicate camera × photo keys in complete master: "
    f"{len(master_duplicates):,}"
)

display(master_duplicates)

Duplicate camera × photo keys in complete master: 22


,CameraAssignment,PhotoNumber,MasterSheet,MasterExcelRow
690,AW120,9174,AW120_Master,692
954,AW120,9174,AW120_Master,956
780,AW120,9435,AW120_Master,782
1040,AW120,9435,AW120_Master,1042
781,AW120,9436,AW120_Master,783
1041,AW120,9436,AW120_Master,1043
782,AW120,9437,AW120_Master,784
1042,AW120,9437,AW120_Master,1044
783,AW120,9438,AW120_Master,785
1043,AW120,9438,AW120_Master,1045


In [44]:
# ============================================================
# INSPECT LEGACY DUPLICATE MASTER ROWS
# Show all metadata fields before Point1
# ============================================================

duplicate_keys = set(
    zip(
        master_duplicates["CameraAssignment"],
        master_duplicates["PhotoNumber"].astype(int)
    )
)

duplicate_detail_frames = []

for sheet_name, camera in [
    ("AW120_Master", "AW120"),
    ("Cam5_Master", "Cam5"),
]:

    df = pd.read_excel(
        PATCHED_MASTER,
        sheet_name=sheet_name
    )

    photo_col = find_df_column(
        df,
        ["Image", "Photo", "Photos"]
    )

    df["CameraAssignment"] = camera
    df["PhotoNumber"] = (
        df[photo_col]
        .map(extract_photo_number)
        .astype("Int64")
    )
    df["MasterSheet"] = sheet_name
    df["MasterExcelRow"] = np.arange(
        2,
        len(df) + 2
    )

    # Everything before Point1 is metadata
    point1_col = find_df_column(
        df,
        ["Point1"]
    )

    point1_position = list(df.columns).index(
        point1_col
    )

    metadata_cols = list(
        df.columns[:point1_position]
    )

    # Add our QA fields
    show_cols = (
        metadata_cols
        + [
            "CameraAssignment",
            "PhotoNumber",
            "MasterSheet",
            "MasterExcelRow",
        ]
    )

    keep = df.apply(
        lambda r: (
            r["PhotoNumber"] is not pd.NA
            and (
                camera,
                int(r["PhotoNumber"])
            ) in duplicate_keys
        ),
        axis=1
    )

    duplicate_detail_frames.append(
        df.loc[keep, show_cols]
    )


duplicate_details = pd.concat(
    duplicate_detail_frames,
    ignore_index=True
)

with pd.option_context(
    "display.max_columns", None,
    "display.max_rows", None,
    "display.max_colwidth", 80,
):
    display(
        duplicate_details.sort_values(
            [
                "CameraAssignment",
                "PhotoNumber",
                "MasterExcelRow",
            ]
        )
    )

,Camera,key,image,Comment,GridSize,CameraAssignment,PhotoNumber,MasterSheet,MasterExcelRow
0,AW120,41,DSCN9174_c.jpg,NaN,100.0,AW120,9174,AW120_Master,692
6,AW120,20,DSCN9174_c.jpg,NaN,100.0,AW120,9174,AW120_Master,956
1,AW120,36,DSCN9435_c.jpg,NaN,NaN,AW120,9435,AW120_Master,782
7,AW120,21,DSCN9435_c.jpg,NaN,100.0,AW120,9435,AW120_Master,1042
2,AW120,37,DSCN9436_c.jpg,NaN,NaN,AW120,9436,AW120_Master,783
8,AW120,22,DSCN9436_c.jpg,NaN,100.0,AW120,9436,AW120_Master,1043
3,AW120,38,DSCN9437_c.jpg,NaN,NaN,AW120,9437,AW120_Master,784
9,AW120,23,DSCN9437_c.jpg,NaN,100.0,AW120,9437,AW120_Master,1044
4,AW120,39,DSCN9438_c.jpg,NaN,NaN,AW120,9438,AW120_Master,785
10,AW120,24,DSCN9438_c.jpg,NaN,100.0,AW120,9438,AW120_Master,1045


In [46]:
# ============================================================
# RECONSTRUCT SOURCE DATABASE PROVENANCE FROM Import_Log
# ============================================================

COMPLETE_MASTER = PATCHED_MASTER

import_log = pd.read_excel(
    COMPLETE_MASTER,
    sheet_name="Import_Log"
)

display(import_log)


def normalize_db(value):

    if pd.isna(value):
        return None

    return (
        str(value)
        .strip()
        .lower()
        .replace(".xls", "")
    )


import_log["DatabaseNorm"] = (
    import_log["Database"]
    .map(normalize_db)
)


# ------------------------------------------------------------
# Build row-range provenance for each camera sheet
# ------------------------------------------------------------

provenance_records = []

for camera, count_col, sheet_name in [
    ("AW120", "RowsToAW120", "AW120_Master"),
    ("Cam5", "RowsToCam5", "Cam5_Master"),
]:

    # Excel row 1 is header, so data begin at row 2
    next_excel_row = 2

    for _, rec in import_log.iterrows():

        n = rec[count_col]

        if pd.isna(n):
            n = 0

        n = int(n)

        if n == 0:
            continue

        start_row = next_excel_row
        end_row = start_row + n - 1

        provenance_records.append({
            "CameraAssignment": camera,
            "MasterSheet": sheet_name,
            "SourceDatabase": rec["Database"],
            "DatabaseNorm": rec["DatabaseNorm"],
            "StartExcelRow": start_row,
            "EndExcelRow": end_row,
            "NRows": n,
        })

        next_excel_row = end_row + 1


provenance = pd.DataFrame(
    provenance_records
)

display(provenance)

,SourceFile,Database,DatabaseAssignment,RowsRead,RowsToAW120,RowsToCam5,Notes
0,AW120_DATABASE_1.XLS,aw120_database_1,AW120,50,50,0,NaN
1,AW120_DATABASE_2.XLS,aw120_database_2,AW120,50,50,0,NaN
2,AW120_DATABASE_3.XLS,aw120_database_3,AW120,50,50,0,NaN
3,AW120_DATABASE_4.XLS,aw120_database_4,AW120,50,50,0,NaN
4,AW120_DATABASE_5.XLS,aw120_database_5,AW120,50,50,0,NaN
5,AW120_DATABASE_6.XLS,aw120_database_6,AW120,50,50,0,NaN
6,AW120_DATABASE_7.XLS,aw120_database_7,AW120,50,50,0,NaN
7,AW120_DATABASE_8.XLS,aw120_database_8,AW120,50,50,0,NaN
8,AW120_DATABASE_9.XLS,aw120_database_9,AW120,50,50,0,NaN
9,AW120_DATABASE_10.XLS,aw120_database_10,AW120,50,50,0,NaN


,CameraAssignment,MasterSheet,SourceDatabase,DatabaseNorm,StartExcelRow,EndExcelRow,NRows
0,AW120,AW120_Master,aw120_database_1,aw120_database_1,2,51,50
1,AW120,AW120_Master,aw120_database_2,aw120_database_2,52,101,50
2,AW120,AW120_Master,aw120_database_3,aw120_database_3,102,151,50
3,AW120,AW120_Master,aw120_database_4,aw120_database_4,152,201,50
4,AW120,AW120_Master,aw120_database_5,aw120_database_5,202,251,50
5,AW120,AW120_Master,aw120_database_6,aw120_database_6,252,301,50
6,AW120,AW120_Master,aw120_database_7,aw120_database_7,302,351,50
7,AW120,AW120_Master,aw120_database_8,aw120_database_8,352,401,50
8,AW120,AW120_Master,aw120_database_9,aw120_database_9,402,451,50
9,AW120,AW120_Master,aw120_database_10,aw120_database_10,452,501,50


In [47]:
# ============================================================
# READ COMPLETE MASTER AND ATTACH SOURCE DATABASE
# ============================================================

master_frames = []

for sheet_name, camera in [
    ("AW120_Master", "AW120"),
    ("Cam5_Master", "Cam5"),
]:

    df = pd.read_excel(
        COMPLETE_MASTER,
        sheet_name=sheet_name
    )

    photo_col = find_df_column(
        df,
        ["Image", "Photo", "Photos"]
    )

    df["CameraAssignment"] = camera
    df["MasterSheet"] = sheet_name
    df["MasterExcelRow"] = np.arange(
        2,
        len(df) + 2
    )

    df["PhotoNumber"] = (
        df[photo_col]
        .map(extract_photo_number)
        .astype("Int64")
    )

    prov = provenance[
        provenance["MasterSheet"] == sheet_name
    ]

    source_db = []

    for excel_row in df["MasterExcelRow"]:

        match = prov[
            (prov["StartExcelRow"] <= excel_row)
            & (prov["EndExcelRow"] >= excel_row)
        ]

        if len(match) != 1:
            raise ValueError(
                f"{sheet_name} row {excel_row}: "
                f"expected exactly one provenance block, "
                f"found {len(match)}."
            )

        source_db.append(
            match.iloc[0]["DatabaseNorm"]
        )

    df["SourceDatabase"] = source_db

    master_frames.append(df)


complete_rows = pd.concat(
    master_frames,
    ignore_index=True
)

print(f"Master rows: {len(complete_rows):,}")
print(
    f"Rows with source DB: "
    f"{complete_rows['SourceDatabase'].notna().sum():,}"
)

display(
    complete_rows[
        [
            "CameraAssignment",
            "PhotoNumber",
            "SourceDatabase",
            "MasterSheet",
            "MasterExcelRow",
        ]
    ].head()
)

Master rows: 2,280
Rows with source DB: 2,280


,CameraAssignment,PhotoNumber,SourceDatabase,MasterSheet,MasterExcelRow
0,AW120,9340,aw120_database_1,AW120_Master,2
1,AW120,9341,aw120_database_1,AW120_Master,3
2,AW120,9342,aw120_database_1,AW120_Master,4
3,AW120,9343,aw120_database_1,AW120_Master,5
4,AW120,9344,aw120_database_1,AW120_Master,6


In [48]:
# ============================================================
# CORRECTED SURVEY = AUTHORITATIVE DATABASE × PHOTO CROSSWALK
# ============================================================

survey = pd.read_excel(
    MASTER_SURVEY,
    sheet_name="Sheet1"
)

PHOTO_COLUMNS = [
    "Center Photoplot",
    "North Photoplot",
    "East Photoplot",
    "South Photoplot",
    "West Photoplot",
]


survey["ResolvedPlotID"] = (
    survey["Plot ID"]
    .where(
        survey["Plot ID"].notna()
        & survey["Plot ID"].astype(str).str.strip().ne(""),
        survey["Plot ID Incidental"]
    )
)

survey["DatabaseNorm"] = (
    survey["Samplepoint Database Name"]
    .map(normalize_db)
)


id_cols = [
    c for c in survey.columns
    if c not in PHOTO_COLUMNS
]


survey_photos = survey.melt(
    id_vars=id_cols,
    value_vars=PHOTO_COLUMNS,
    var_name="PhotoPosition",
    value_name="SurveyPhoto",
)


survey_photos["PhotoNumber"] = (
    survey_photos["SurveyPhoto"]
    .map(extract_photo_number)
    .astype("Int64")
)


survey_photos = (
    survey_photos[
        survey_photos["PhotoNumber"].notna()
        & survey_photos["DatabaseNorm"].notna()
    ]
    .copy()
)


# Database + photo is the correction key.
survey_valid_keys = set(
    zip(
        survey_photos["DatabaseNorm"],
        survey_photos["PhotoNumber"].astype(int)
    )
)


print(
    f"Survey photo assignments with database: "
    f"{len(survey_photos):,}"
)

print(
    f"Unique database × photo keys: "
    f"{len(survey_valid_keys):,}"
)

Survey photo assignments with database: 2,288
Unique database × photo keys: 2,288


In [49]:
# ============================================================
# TRANSFER CORRECTED SURVEY DECISIONS TO RECOVERED MASTER
# ============================================================

complete_rows["SurveyKeyValid"] = [
    (
        db,
        int(photo)
    ) in survey_valid_keys
    if pd.notna(photo) and db is not None
    else False

    for db, photo in zip(
        complete_rows["SourceDatabase"],
        complete_rows["PhotoNumber"],
    )
]


valid_rows = (
    complete_rows[
        complete_rows["SurveyKeyValid"]
    ]
    .copy()
)

stale_rows = (
    complete_rows[
        ~complete_rows["SurveyKeyValid"]
    ]
    .copy()
)


print("=" * 72)
print("CORRECTED SURVEY TRANSFER")
print("=" * 72)

print(
    f"Current completed rows: "
    f"{len(complete_rows):,}"
)

print(
    f"KEEP under corrected survey: "
    f"{len(valid_rows):,}"
)

print(
    f"DROP as stale old assignments: "
    f"{len(stale_rows):,}"
)


display(
    stale_rows[
        [
            "CameraAssignment",
            "PhotoNumber",
            "SourceDatabase",
            "MasterSheet",
            "MasterExcelRow",
        ]
    ].sort_values(
        [
            "CameraAssignment",
            "PhotoNumber",
        ]
    )
)

CORRECTED SURVEY TRANSFER
Current completed rows: 2,280
KEEP under corrected survey: 2,268
DROP as stale old assignments: 12


,CameraAssignment,PhotoNumber,SourceDatabase,MasterSheet,MasterExcelRow
954,AW120,9174,garrett_database_5,AW120_Master,956
780,AW120,9435,chloe_database_5,AW120_Master,782
781,AW120,9436,chloe_database_5,AW120_Master,783
782,AW120,9437,chloe_database_5,AW120_Master,784
783,AW120,9438,chloe_database_5,AW120_Master,785
784,AW120,9439,chloe_database_5,AW120_Master,786
394,AW120,9915,aw120_database_8,AW120_Master,396
1965,Cam5,9161,cam5_database_15,Cam5_Master,702
1966,Cam5,9162,cam5_database_15,Cam5_Master,703
1967,Cam5,9163,cam5_database_15,Cam5_Master,704


In [50]:
# ============================================================
# MASTER SURVEY WINS
#
# 2026 Master Survey.xlsx is authoritative for row identity.
# RECOVERED_COMPLETE supplies classification data only.
# ============================================================

# Databases actually represented in the SamplePoint master.
# This intentionally excludes things such as calibration records
# that have no SamplePoint source database.
represented_databases = set(
    complete_rows[
        "SourceDatabase"
    ].dropna()
)


# Corrected survey rows relevant to these SamplePoint databases
survey_expected = (
    survey_photos[
        survey_photos[
            "DatabaseNorm"
        ].isin(represented_databases)
    ]
    [
        [
            "DatabaseNorm",
            "PhotoNumber",
            "ResolvedPlotID",
            "PhotoPosition",
        ]
    ]
    .copy()
)


# ------------------------------------------------------------
# QA corrected survey itself
# ------------------------------------------------------------

survey_dupes = (
    survey_expected[
        survey_expected.duplicated(
            [
                "DatabaseNorm",
                "PhotoNumber",
            ],
            keep=False
        )
    ]
    .sort_values(
        [
            "DatabaseNorm",
            "PhotoNumber",
        ]
    )
)

if len(survey_dupes):
    display(survey_dupes)

    raise ValueError(
        "Corrected 2026 Master Survey still contains "
        "duplicate database × photo assignments."
    )


# ------------------------------------------------------------
# QA patched classification inventory
# ------------------------------------------------------------

patched_dupes = (
    complete_rows[
        complete_rows.duplicated(
            [
                "SourceDatabase",
                "PhotoNumber",
            ],
            keep=False
        )
    ]
    .sort_values(
        [
            "SourceDatabase",
            "PhotoNumber",
        ]
    )
)

if len(patched_dupes):
    print(
        "Duplicate database × photo rows exist "
        "inside patched master:"
    )

    display(
        patched_dupes[
            [
                "SourceDatabase",
                "PhotoNumber",
                "CameraAssignment",
                "MasterSheet",
                "MasterExcelRow",
            ]
        ]
    )


# ------------------------------------------------------------
# Compare sets
# ------------------------------------------------------------

survey_keys = set(
    zip(
        survey_expected["DatabaseNorm"],
        survey_expected["PhotoNumber"].astype(int),
    )
)

patched_keys = set(
    zip(
        complete_rows["SourceDatabase"],
        complete_rows["PhotoNumber"].astype(int),
    )
)


stale_keys = (
    patched_keys
    - survey_keys
)

missing_keys = (
    survey_keys
    - patched_keys
)


complete_rows["KeepByCorrectedSurvey"] = [
    (
        db,
        int(photo)
    ) in survey_keys

    for db, photo in zip(
        complete_rows["SourceDatabase"],
        complete_rows["PhotoNumber"],
    )
]


stale_rows = (
    complete_rows[
        ~complete_rows[
            "KeepByCorrectedSurvey"
        ]
    ]
    .copy()
)


print("=" * 72)
print("CORRECTED MASTER SURVEY RECONCILIATION")
print("=" * 72)

print(
    f"Patched classification rows: "
    f"{len(complete_rows):,}"
)

print(
    f"Corrected survey rows:       "
    f"{len(survey_expected):,}"
)

print(
    f"Rows accepted:               "
    f"{complete_rows['KeepByCorrectedSurvey'].sum():,}"
)

print(
    f"Rows rejected as stale:      "
    f"{len(stale_rows):,}"
)

print(
    f"Survey keys missing data:    "
    f"{len(missing_keys):,}"
)


print("\nROWS REJECTED BECAUSE SURVEY MASTER DISAGREES:")

display(
    stale_rows[
        [
            "SourceDatabase",
            "PhotoNumber",
            "CameraAssignment",
            "MasterSheet",
            "MasterExcelRow",
        ]
    ]
    .sort_values(
        [
            "SourceDatabase",
            "PhotoNumber",
        ]
    )
)


if missing_keys:

    print(
        "\nSURVEY MASTER EXPECTS THESE "
        "BUT CLASSIFICATION MASTER LACKS THEM:"
    )

    display(
        pd.DataFrame(
            sorted(missing_keys),
            columns=[
                "DatabaseNorm",
                "PhotoNumber",
            ]
        )
    )

    raise ValueError(
        "The corrected survey contains expected photos "
        "with no classification row. Do not write the "
        "final master until these are resolved."
    )

CORRECTED MASTER SURVEY RECONCILIATION
Patched classification rows: 2,280
Corrected survey rows:       2,268
Rows accepted:               2,268
Rows rejected as stale:      12
Survey keys missing data:    0

ROWS REJECTED BECAUSE SURVEY MASTER DISAGREES:


,SourceDatabase,PhotoNumber,CameraAssignment,MasterSheet,MasterExcelRow
394,aw120_database_8,9915,AW120,AW120_Master,396
1965,cam5_database_15,9161,Cam5,Cam5_Master,702
1966,cam5_database_15,9162,Cam5,Cam5_Master,703
1967,cam5_database_15,9163,Cam5,Cam5_Master,704
1968,cam5_database_15,9164,Cam5,Cam5_Master,705
1969,cam5_database_15,9165,Cam5,Cam5_Master,706
780,chloe_database_5,9435,AW120,AW120_Master,782
781,chloe_database_5,9436,AW120,AW120_Master,783
782,chloe_database_5,9437,AW120,AW120_Master,784
783,chloe_database_5,9438,AW120,AW120_Master,785


In [53]:
# ============================================================
# CAMERA NORMALIZATION HELPERS
# ============================================================

OBSERVER_DB_CAMERA = {
    "chloe_database_2": "AW120",
    "chloe_database_5": "AW120",
    "garrett_database_1": "AW120",
    "garrett_database_2": "AW120",
    "garrett_database_3": "AW120",
    "garrett_database_4": "AW120",
    "garrett_database_5": "AW120",
    "garrett_database_8": "AW120",
    "janelle_database_1": "AW120",
    "janelle_database_2": "AW120",
    "janelle_database_3": "AW120",
    "janelle_database_4": "AW120",

    "chloe_database_3": "Cam5",
    "garrett_database_6": "Cam5",
    "janelle_database_6": "Cam5",
}


def normalize_camera(value):
    """
    Normalize the Camera Number field from the corrected
    2026 Master Survey.
    """

    if value is None:
        return None

    try:
        if pd.isna(value):
            return None
    except (TypeError, ValueError):
        pass

    text = str(value).strip().lower()

    if "aw120" in text:
        return "AW120"

    if (
        "cam5" in text
        or "cam 5" in text
        or "camera5" in text
        or "camera 5" in text
    ):
        return "Cam5"

    return None


def camera_from_database(value):
    """
    Infer camera only for databases that are unambiguously
    associated with one camera.

    Mixed-camera databases intentionally return None so that
    the corrected survey determines their camera assignment.
    """

    if value is None:
        return None

    try:
        if pd.isna(value):
            return None
    except (TypeError, ValueError):
        pass

    db = normalize_db(value)

    if db is None:
        return None

    if db.startswith("aw120_database_"):
        return "AW120"

    if db.startswith("cam5_database_"):
        return "Cam5"

    return OBSERVER_DB_CAMERA.get(db)

In [57]:
# ============================================================
# REFERENCE RECOVERED_COMPLETE AGAINST CORRECTED 2026 SURVEY
#
# AUTHORITATIVE CAMERA RULES
#   2026 Master Survey.xlsx:
#       use "Camera Number"
#
#   RECOVERED_COMPLETE.xlsx:
#       sheet 1 = AW120
#       sheet 2 = Cam5
#
# MATCH KEY:
#       Camera + PhotoNumber
#
# This cell modifies NOTHING.
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd


MASTER_SURVEY = Path(
    r"D:\My Drive\BOP_OCTC_2025\2026 Master Survey.xlsx"
)

RECOVERED_COMPLETE = Path(
    r"D:\My Drive\BOP_OCTC_2025"
    r"\Photos\2026\Cropped\XLS_OutputFiles"
    r"\2026_SamplePoint_Camera_Masters_RECOVERED_COMPLETE.xlsx"
)


PHOTO_COLUMNS = [
    "Center Photoplot",
    "North Photoplot",
    "East Photoplot",
    "South Photoplot",
    "West Photoplot",
]


# ------------------------------------------------------------
# CAMERA NORMALIZATION
# ------------------------------------------------------------

def normalize_camera(value):

    if value is None:
        return None

    try:
        if pd.isna(value):
            return None
    except (TypeError, ValueError):
        pass

    text = str(value).strip().lower()

    if "aw120" in text:
        return "AW120"

    if (
        "cam5" in text
        or "cam 5" in text
        or "camera5" in text
        or "camera 5" in text
    ):
        return "Cam5"

    return None


# ============================================================
# 1. READ CORRECTED 2026 MASTER SURVEY
# ============================================================

survey = pd.read_excel(
    MASTER_SURVEY,
    sheet_name="Sheet1"
).copy()


# Use Plot ID when present;
# otherwise use Plot ID Incidental.
survey["ResolvedPlotID"] = (
    survey["Plot ID"]
    .where(
        survey["Plot ID"].notna()
        & survey["Plot ID"]
            .astype(str)
            .str.strip()
            .ne(""),
        survey["Plot ID Incidental"]
    )
)


survey["CameraAssignment"] = (
    survey["Camera Number"]
    .map(normalize_camera)
)


print("=" * 72)
print("CORRECTED 2026 SURVEY CAMERA STATUS")
print("=" * 72)

display(
    survey["CameraAssignment"]
    .value_counts(dropna=False)
    .rename("SurveySites")
    .to_frame()
)


# ============================================================
# 2. EXPAND THE FIVE DEFINITIVE PHOTO FIELDS
# ============================================================

survey_photo_records = []


for _, site in survey.iterrows():

    camera = site["CameraAssignment"]

    for photo_position in PHOTO_COLUMNS:

        photo_number = extract_photo_number(
            site[photo_position]
        )

        if photo_number is None:
            continue

        survey_photo_records.append({

            "CameraAssignment":
                camera,

            "PhotoNumber":
                int(photo_number),

            "ResolvedPlotID":
                site["ResolvedPlotID"],

            "PhotoPosition":
                photo_position,

            "Samplepoint Database Name":
                site["Samplepoint Database Name"],

            # Carry the complete definitive site photo record
            "Center Photoplot":
                site["Center Photoplot"],

            "North Photoplot":
                site["North Photoplot"],

            "East Photoplot":
                site["East Photoplot"],

            "South Photoplot":
                site["South Photoplot"],

            "West Photoplot":
                site["West Photoplot"],
        })


survey_photos = pd.DataFrame(
    survey_photo_records
)


# Remove exact duplicate records only.
survey_photos = (
    survey_photos
    .drop_duplicates()
    .reset_index(drop=True)
)


# ============================================================
# 3. TRUE CAMERA-AWARE CONFLICT CHECK
# ============================================================

resolved_survey_photos = (
    survey_photos[
        survey_photos["CameraAssignment"].notna()
    ]
    .copy()
)


camera_photo_site = (
    resolved_survey_photos[
        [
            "CameraAssignment",
            "PhotoNumber",
            "ResolvedPlotID",
        ]
    ]
    .drop_duplicates()
)


conflicting_camera_photos = (
    camera_photo_site[
        camera_photo_site.duplicated(
            [
                "CameraAssignment",
                "PhotoNumber",
            ],
            keep=False
        )
    ]
    .sort_values(
        [
            "CameraAssignment",
            "PhotoNumber",
            "ResolvedPlotID",
        ]
    )
)


print("\n" + "=" * 72)
print("CAMERA-AWARE SURVEY PHOTO QA")
print("=" * 72)

print(
    f"Survey photo records:                  "
    f"{len(survey_photos):,}"
)

print(
    f"Photos with resolved camera:           "
    f"{len(resolved_survey_photos):,}"
)

print(
    f"Photos with unresolved camera:         "
    f"{survey_photos['CameraAssignment'].isna().sum():,}"
)

print(
    f"True camera × photo site conflicts:    "
    f"{len(conflicting_camera_photos):,}"
)


if len(conflicting_camera_photos):

    print(
        "\nTRUE within-camera conflicts:"
    )

    display(
        conflicting_camera_photos
    )

    raise ValueError(
        "Corrected survey still contains "
        "camera + photo numbers assigned to "
        "more than one site."
    )


# One definitive survey record per camera + photo
photo_lookup = (
    resolved_survey_photos
    .drop_duplicates(
        subset=[
            "CameraAssignment",
            "PhotoNumber",
        ]
    )
    .copy()
)


# ============================================================
# 4. READ RECOVERED_COMPLETE
#
# Sheet 1 = AW120
# Sheet 2 = Cam5
# ============================================================

xls = pd.ExcelFile(
    RECOVERED_COMPLETE
)


assert len(xls.sheet_names) >= 2


AW120_SHEET = xls.sheet_names[0]
CAM5_SHEET = xls.sheet_names[1]


print("\n" + "=" * 72)
print("RECOVERED COMPLETE SHEETS")
print("=" * 72)

print(
    f"Sheet 1 → AW120: {AW120_SHEET}"
)

print(
    f"Sheet 2 → Cam5:  {CAM5_SHEET}"
)


recovered_frames = []


for sheet_name, camera in [
    (AW120_SHEET, "AW120"),
    (CAM5_SHEET, "Cam5"),
]:

    df = pd.read_excel(
        RECOVERED_COMPLETE,
        sheet_name=sheet_name
    )

    photo_col = find_df_column(
        df,
        ["Image", "Photo", "Photos"]
    )

    df["CameraAssignment"] = camera

    df["MasterSheet"] = sheet_name

    df["MasterExcelRow"] = np.arange(
        2,
        len(df) + 2
    )

    df["PhotoNumber"] = (
        df[photo_col]
        .map(extract_photo_number)
        .astype("Int64")
    )

    recovered_frames.append(df)


recovered = pd.concat(
    recovered_frames,
    ignore_index=True
)


print("\nRecovered rows by camera:")

display(
    recovered["CameraAssignment"]
    .value_counts()
    .rename("Photos")
    .to_frame()
)


# ============================================================
# 5. CAMERA + PHOTO MATCH TO CORRECTED SURVEY
# ============================================================

recovered_with_site = (
    recovered
    .merge(
        photo_lookup,
        on=[
            "CameraAssignment",
            "PhotoNumber",
        ],
        how="left",
        validate="many_to_one",
        indicator=True
    )
)


matched = (
    recovered_with_site["_merge"] == "both"
)

unmatched = ~matched


print("\n" + "=" * 72)
print("RECOVERED COMPLETE ↔ CORRECTED 2026 SURVEY")
print("KEY = CAMERA + PHOTO NUMBER")
print("=" * 72)

print(
    f"Recovered classification rows: "
    f"{len(recovered_with_site):,}"
)

print(
    f"Matched to corrected survey:   "
    f"{matched.sum():,}"
)

print(
    f"Not matched:                   "
    f"{unmatched.sum():,}"
)

print(
    f"Corrected sites represented:   "
    f"{recovered_with_site.loc[matched, 'ResolvedPlotID'].nunique():,}"
)


# ============================================================
# 6. SHOW ACTUAL UNMATCHED ROWS, IF ANY
# ============================================================

if unmatched.any():

    print(
        "\nRecovered photos not found in "
        "the corrected survey:"
    )

    display(
        recovered_with_site.loc[
            unmatched,
            [
                "CameraAssignment",
                "PhotoNumber",
                "MasterSheet",
                "MasterExcelRow",
            ]
        ]
        .sort_values(
            [
                "CameraAssignment",
                "PhotoNumber",
            ]
        )
        .reset_index(drop=True)
    )


# ============================================================
# 7. CORRECTED SITES REPRESENTED IN RECOVERED_COMPLETE
# ============================================================

sites_present = (
    recovered_with_site.loc[
        matched,
        [
            "ResolvedPlotID",
            "Samplepoint Database Name",
            "CameraAssignment",
            "Center Photoplot",
            "North Photoplot",
            "East Photoplot",
            "South Photoplot",
            "West Photoplot",
        ]
    ]
    .drop_duplicates()
    .sort_values(
        [
            "CameraAssignment",
            "ResolvedPlotID",
        ]
    )
    .reset_index(drop=True)
)


print(
    f"\nUnique corrected site records represented: "
    f"{len(sites_present):,}"
)

display(sites_present)

CORRECTED 2026 SURVEY CAMERA STATUS


,SurveySites
CameraAssignment,
AW120,255
Cam5,202
NaN,144



CAMERA-AWARE SURVEY PHOTO QA
Survey photo records:                  2,288
Photos with resolved camera:           2,283
Photos with unresolved camera:         5
True camera × photo site conflicts:    0

RECOVERED COMPLETE SHEETS
Sheet 1 → AW120: AW120_Master
Sheet 2 → Cam5:  Cam5_Master

Recovered rows by camera:


,Photos
CameraAssignment,
AW120,1265
Cam5,1015



RECOVERED COMPLETE ↔ CORRECTED 2026 SURVEY
KEY = CAMERA + PHOTO NUMBER
Recovered classification rows: 2,280
Matched to corrected survey:   2,274
Not matched:                   6
Corrected sites represented:   442

Recovered photos not found in the corrected survey:


,CameraAssignment,PhotoNumber,MasterSheet,MasterExcelRow
0,AW120,9295,AW120_Master,627
1,AW120,9296,AW120_Master,628
2,AW120,9297,AW120_Master,629
3,AW120,9298,AW120_Master,630
4,AW120,9299,AW120_Master,631
5,AW120,9915,AW120_Master,396



Unique corrected site records represented: 453


,ResolvedPlotID,Samplepoint Database Name,CameraAssignment,Center Photoplot,North Photoplot,East Photoplot,South Photoplot,West Photoplot
0,20260603_low76_fourwing1,AW120_database_1,AW120,9345.0,9346.0,9347.0,9348.0,9349.0
1,20260603_low76_fourwing2,AW120_database_5,AW120,9355.0,9356.0,9357.0,9358.0,9359.0
2,20260604_Mid_21_DISPland,garrett_database_8,AW120,9370.0,9371.0,9372.0,9373.0,9374.0
3,20260604_mid_21_savebc,AW120_database_13,AW120,9395.0,9396.0,9397.0,9398.0,9399.0
4,20260611_low_69_HAGL,AW120_database_1,AW120,9555.0,9556.0,9557.0,9558.0,9559.0
...,...,...,...,...,...,...,...,...
448,mid_8_12,Cam5_Database_4,Cam5,8633.0,8634.0,8635.0,8636.0,8637.0
449,mid_8_21,Cam5_Database_4,Cam5,8618.0,8619.0,8620.0,8621.0,8622.0
450,mid_8_22,Cam5_Database_4,Cam5,8638.0,8639.0,8640.0,8641.0,8642.0
451,mid_8_31,Cam5_Database_4,Cam5,8623.0,8624.0,8625.0,8626.0,8627.0


In [58]:
# ============================================================
# CREATE RGB-ONLY AND TEXT-ONLY PRODUCTS
# FROM RECOVERED_COMPLETE
#
# Source workbook remains untouched.
# Only Point1 ... Point100 are transformed.
# ============================================================

from pathlib import Path
import re
import shutil
import openpyxl


MASTER_XLSX = Path(
    r"D:\My Drive\BOP_OCTC_2025"
    r"\Photos\2026\Cropped\XLS_OutputFiles"
    r"\2026_SamplePoint_Camera_Masters_RECOVERED_COMPLETE.xlsx"
)


RGB_XLSX = MASTER_XLSX.with_name(
    "2026_SamplePoint_Camera_Masters_RECOVERED_COMPLETE_RGBonly.xlsx"
)

TEXT_XLSX = MASTER_XLSX.with_name(
    "2026_SamplePoint_Camera_Masters_RECOVERED_COMPLETE_TextOnly.xlsx"
)


# ------------------------------------------------------------
# SamplePoint values look like:
#
#     OTHER, 133, 140, 122
#     DEPI, 84, 101, 72
#     ROCK, 155, 149, 136
#
# Interpret the LAST three comma-separated integers as RGB.
# Everything before them is the classification label.
# ------------------------------------------------------------

RGB_RE = re.compile(
    r"^(.*?)\s*,\s*"
    r"(\d{1,3})\s*,\s*"
    r"(\d{1,3})\s*,\s*"
    r"(\d{1,3})\s*$"
)


def split_samplepoint_value(value):

    if value is None:
        return None, None

    text = str(value).strip()

    if not text:
        return None, None

    match = RGB_RE.match(text)

    if match is None:
        return None, None

    label = match.group(1).strip()

    r = int(match.group(2))
    g = int(match.group(3))
    b = int(match.group(4))

    if not all(
        0 <= x <= 255
        for x in (r, g, b)
    ):
        return None, None

    rgb = f"{r}, {g}, {b}"

    return label, rgb


# ------------------------------------------------------------
# Make exact copies first
# ------------------------------------------------------------

shutil.copy2(
    MASTER_XLSX,
    RGB_XLSX
)

shutil.copy2(
    MASTER_XLSX,
    TEXT_XLSX
)


# ------------------------------------------------------------
# Transform Point1 ... Point100 only
#
# Sheet 1 = AW120
# Sheet 2 = Cam5
# ------------------------------------------------------------

def transform_workbook(path, mode):

    wb = openpyxl.load_workbook(path)

    transformed = 0
    blank = 0
    unresolved = []

    # Explicitly use first two sheets
    target_sheets = [
        wb.sheetnames[0],  # AW120
        wb.sheetnames[1],  # Cam5
    ]

    for sheet_name in target_sheets:

        ws = wb[sheet_name]

        headers = {
            str(cell.value).strip(): cell.column
            for cell in ws[1]
            if cell.value is not None
        }

        point_cols = {
            n: headers[f"Point{n}"]
            for n in range(1, 101)
        }

        for excel_row in range(
            2,
            ws.max_row + 1
        ):

            for point_number, col in point_cols.items():

                cell = ws.cell(
                    excel_row,
                    col
                )

                original = cell.value

                if (
                    original is None
                    or str(original).strip() == ""
                ):
                    blank += 1
                    continue

                label, rgb = (
                    split_samplepoint_value(
                        original
                    )
                )

                if label is None:

                    unresolved.append({
                        "Sheet": sheet_name,
                        "ExcelRow": excel_row,
                        "Point": point_number,
                        "Value": original,
                    })

                    continue

                if mode == "rgb":
                    cell.value = rgb

                elif mode == "text":
                    cell.value = label

                else:
                    raise ValueError(
                        "mode must be 'rgb' or 'text'"
                    )

                transformed += 1

    wb.save(path)
    wb.close()

    return (
        transformed,
        blank,
        unresolved,
    )


# ------------------------------------------------------------
# CREATE RGB PRODUCT
# ------------------------------------------------------------

rgb_n, rgb_blank, rgb_unresolved = (
    transform_workbook(
        RGB_XLSX,
        mode="rgb"
    )
)


# ------------------------------------------------------------
# CREATE TEXT PRODUCT
# ------------------------------------------------------------

text_n, text_blank, text_unresolved = (
    transform_workbook(
        TEXT_XLSX,
        mode="text"
    )
)


# ------------------------------------------------------------
# QA
# ------------------------------------------------------------

print("=" * 72)
print("SAMPLEPOINT DATASETS CREATED")
print("=" * 72)

print(
    f"RGB-only Point cells transformed:  "
    f"{rgb_n:,}"
)

print(
    f"Text-only Point cells transformed: "
    f"{text_n:,}"
)

print(
    f"\nBlank RGB cells:  "
    f"{rgb_blank:,}"
)

print(
    f"Blank Text cells: "
    f"{text_blank:,}"
)

print(
    f"\nUnresolved RGB cells:  "
    f"{len(rgb_unresolved):,}"
)

print(
    f"Unresolved Text cells: "
    f"{len(text_unresolved):,}"
)

print(
    f"\nRGB-only:\n{RGB_XLSX}"
)

print(
    f"\nText-only:\n{TEXT_XLSX}"
)


# ------------------------------------------------------------
# EXPECTED FINAL COUNTS
# ------------------------------------------------------------

expected_points = 2280 * 100

assert rgb_n == expected_points
assert text_n == expected_points
assert rgb_blank == 0
assert text_blank == 0
assert len(rgb_unresolved) == 0
assert len(text_unresolved) == 0

print(
    "\nPASS: all 228,000 Point cells "
    "were successfully split."
)

SAMPLEPOINT DATASETS CREATED
RGB-only Point cells transformed:  228,000
Text-only Point cells transformed: 228,000

Blank RGB cells:  0
Blank Text cells: 0

Unresolved RGB cells:  0
Unresolved Text cells: 0

RGB-only:
D:\My Drive\BOP_OCTC_2025\Photos\2026\Cropped\XLS_OutputFiles\2026_SamplePoint_Camera_Masters_RECOVERED_COMPLETE_RGBonly.xlsx

Text-only:
D:\My Drive\BOP_OCTC_2025\Photos\2026\Cropped\XLS_OutputFiles\2026_SamplePoint_Camera_Masters_RECOVERED_COMPLETE_TextOnly.xlsx

PASS: all 228,000 Point cells were successfully split.


In [59]:
# ============================================================
# AUDIT COMMENTS ASSOCIATED WITH OTHER HITS
#
# Run AFTER creating:
#   ...RECOVERED_COMPLETE_TextOnly.xlsx
#
# Nothing is modified.
# ============================================================

import pandas as pd
import numpy as np


# TEXT_XLSX should already exist from the split cell.
print(f"Reading:\n{TEXT_XLSX}")


xls = pd.ExcelFile(TEXT_XLSX)

assert len(xls.sheet_names) >= 2

AW120_SHEET = xls.sheet_names[0]
CAM5_SHEET = xls.sheet_names[1]


other_row_records = []
all_comment_records = []


for sheet_name, camera in [
    (AW120_SHEET, "AW120"),
    (CAM5_SHEET, "Cam5"),
]:

    df = pd.read_excel(
        TEXT_XLSX,
        sheet_name=sheet_name
    )

    # --------------------------------------------------------
    # Find metadata fields
    # --------------------------------------------------------

    photo_col = find_df_column(
        df,
        ["Image", "Photo", "Photos"]
    )

    comment_col = find_df_column(
        df,
        ["Comment", "Comments"]
    )

    point_cols = [
        find_df_column(
            df,
            [f"Point{n}"]
        )
        for n in range(1, 101)
    ]


    # --------------------------------------------------------
    # Normalize comments for auditing only
    # Keep original text too.
    # --------------------------------------------------------

    comments = (
        df[comment_col]
        .fillna("")
        .astype(str)
        .str.strip()
    )


    # --------------------------------------------------------
    # Count literal OTHER hits per photo
    # --------------------------------------------------------

    point_text = (
        df[point_cols]
        .fillna("")
        .astype(str)
        .apply(
            lambda col: col.str.strip()
        )
    )

    other_mask = (
        point_text.eq("OTHER")
    )

    other_count = (
        other_mask.sum(axis=1)
    )


    # --------------------------------------------------------
    # Record every nonblank comment
    # --------------------------------------------------------

    for idx in df.index:

        comment = comments.loc[idx]

        if comment == "":
            continue

        all_comment_records.append({
            "Camera": camera,
            "Sheet": sheet_name,
            "ExcelRow": int(idx) + 2,
            "PhotoNumber": extract_photo_number(
                df.loc[idx, photo_col]
            ),
            "Comment": comment,
            "OTHER_hits": int(
                other_count.loc[idx]
            ),
        })


    # --------------------------------------------------------
    # Record every row containing >= 1 OTHER
    # --------------------------------------------------------

    rows_with_other = (
        other_count > 0
    )

    for idx in df.index[rows_with_other]:

        # Which point numbers are OTHER?
        other_points = [
            n
            for n, col in enumerate(
                point_cols,
                start=1
            )
            if other_mask.loc[idx, col]
        ]

        other_row_records.append({
            "Camera": camera,
            "Sheet": sheet_name,
            "ExcelRow": int(idx) + 2,
            "PhotoNumber": extract_photo_number(
                df.loc[idx, photo_col]
            ),
            "Comment": comments.loc[idx],
            "OTHER_hits": int(
                other_count.loc[idx]
            ),
            "OTHER_points": ", ".join(
                map(str, other_points)
            ),
        })


other_rows = pd.DataFrame(
    other_row_records
)

all_comments = pd.DataFrame(
    all_comment_records
)


# ============================================================
# SUMMARY
# ============================================================

print("=" * 72)
print("OTHER + COMMENT AUDIT")
print("=" * 72)

print(
    f"Photo rows containing OTHER: "
    f"{len(other_rows):,}"
)

print(
    f"Total literal OTHER hits:    "
    f"{other_rows['OTHER_hits'].sum():,}"
)

print(
    f"OTHER rows with comment:      "
    f"{other_rows['Comment'].ne('').sum():,}"
)

print(
    f"OTHER rows with NO comment:   "
    f"{other_rows['Comment'].eq('').sum():,}"
)


# ============================================================
# UNIQUE COMMENT PATTERNS ON ROWS CONTAINING OTHER
# ============================================================

other_comment_patterns = (
    other_rows
    .groupby(
        [
            "Camera",
            "Comment",
        ],
        dropna=False
    )
    .agg(
        PhotoRows=(
            "PhotoNumber",
            "size"
        ),
        OTHER_hits=(
            "OTHER_hits",
            "sum"
        ),
    )
    .reset_index()
    .sort_values(
        [
            "OTHER_hits",
            "PhotoRows",
            "Camera",
            "Comment",
        ],
        ascending=[
            False,
            False,
            True,
            True,
        ]
    )
    .reset_index(drop=True)
)


print(
    "\nUNIQUE COMMENTS ON ROWS CONTAINING OTHER"
)

with pd.option_context(
    "display.max_rows", None,
    "display.max_colwidth", None,
):
    display(
        other_comment_patterns
    )


# ============================================================
# SHOW ROW-LEVEL DETAIL
# ============================================================

print(
    "\nROW-LEVEL OTHER DETAIL"
)

with pd.option_context(
    "display.max_rows", None,
    "display.max_colwidth", None,
):
    display(
        other_rows.sort_values(
            [
                "Camera",
                "Comment",
                "PhotoNumber",
            ]
        )
    )


# ============================================================
# ALSO SHOW ALL UNIQUE COMMENTS,
# INCLUDING COMMENTS ON ROWS WITHOUT OTHER
# ============================================================

all_unique_comments = (
    all_comments
    .groupby(
        [
            "Camera",
            "Comment",
        ]
    )
    .agg(
        PhotoRows=(
            "PhotoNumber",
            "size"
        ),
        OTHER_hits=(
            "OTHER_hits",
            "sum"
        ),
    )
    .reset_index()
    .sort_values(
        [
            "OTHER_hits",
            "PhotoRows",
            "Camera",
            "Comment",
        ],
        ascending=[
            False,
            False,
            True,
            True,
        ]
    )
    .reset_index(drop=True)
)


print(
    "\nALL UNIQUE COMMENTS"
)

with pd.option_context(
    "display.max_rows", None,
    "display.max_colwidth", None,
):
    display(
        all_unique_comments
    )

Reading:
D:\My Drive\BOP_OCTC_2025\Photos\2026\Cropped\XLS_OutputFiles\2026_SamplePoint_Camera_Masters_RECOVERED_COMPLETE_TextOnly.xlsx
OTHER + COMMENT AUDIT
Photo rows containing OTHER: 222
Total literal OTHER hits:    4,765
OTHER rows with comment:      213
OTHER rows with NO comment:   9

UNIQUE COMMENTS ON ROWS CONTAINING OTHER


,Camera,Comment,PhotoRows,OTHER_hits
0,Cam5,all other hits are ERTR13,22,648
1,Cam5,all other hits are SAVE4,16,603
2,AW120,all others are SAVE4,8,412
3,AW120,all others are HECO26,15,323
4,Cam5,other points; SAVE4,5,274
5,Cam5,Other points; SAVE4,7,219
6,Cam5,Other points; ATCO,4,218
7,AW120,all other hits are ARAR8,4,182
8,AW120,all other hits are ERNA10,3,162
9,AW120,all other hits are DEPI,22,104



ROW-LEVEL OTHER DETAIL


,Camera,Sheet,ExcelRow,PhotoNumber,Comment,OTHER_hits,OTHER_points
31,AW120,AW120_Master,149,9683,,1,2
45,AW120,AW120_Master,402,9921,All others are DEPI,9,"46, 53, 54, 56, 68, 77, 78, 79, 80"
100,AW120,AW120_Master,1032,9370,All others are DISP,90,"1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 21, 22, 23, 24, 25, 26, 27, 28, 31, 34, 35, 36, 38, 39, 41, 43, 44, 45, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100"
72,AW120,AW120_Master,789,9452,aall other hits are DEPI,6,"42, 51, 54, 61, 62, 63"
89,AW120,AW120_Master,904,9066,all other are DEPI,5,"47, 57, 58, 59, 68"
24,AW120,AW120_Master,129,9663,"all other are HECO26 except pts 21,42, are CHJU",10,"2, 10, 20, 21, 26, 30, 42, 63, 70, 73"
95,AW120,AW120_Master,1027,9365,all other are Save4,94,"1, 2, 3, 4, 5, 6, 7, 8, 9, 11, 12, 13, 14, 15, 16, 17, 18, 19, 21, 22, 23, 24, 25, 26, 27, 28, 31, 32, 33, 34, 35, 36, 37, 38, 39, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100"
39,AW120,AW120_Master,302,9821,all other hits are ARAR8,25,"1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 23, 24, 25, 26, 27, 28, 29"
40,AW120,AW120_Master,303,9822,all other hits are ARAR8,42,"1, 2, 3, 4, 5, 6, 7, 11, 12, 13, 14, 15, 21, 22, 23, 24, 31, 32, 33, 41, 42, 43, 51, 52, 53, 54, 61, 62, 63, 72, 73, 74, 75, 82, 83, 84, 85, 91, 92, 93, 94, 95"
41,AW120,AW120_Master,304,9823,all other hits are ARAR8,63,"16, 17, 18, 23, 24, 25, 26, 27, 28, 29, 34, 35, 36, 37, 38, 39, 40, 44, 45, 46, 47, 48, 49, 50, 54, 55, 56, 57, 58, 59, 60, 64, 65, 66, 67, 68, 69, 70, 74, 75, 76, 77, 78, 79, 80, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100"



ALL UNIQUE COMMENTS


,Camera,Comment,PhotoRows,OTHER_hits
0,Cam5,all other hits are ERTR13,22,648
1,Cam5,all other hits are SAVE4,16,603
2,AW120,all others are SAVE4,8,412
3,AW120,all others are HECO26,15,323
4,Cam5,other points; SAVE4,5,274
5,Cam5,Other points; SAVE4,7,219
6,Cam5,Other points; ATCO,4,218
7,AW120,all other hits are ARAR8,4,182
8,AW120,all other hits are ERNA10,3,162
9,AW120,all other hits are DEPI,22,104


In [60]:
# ============================================================
# DRY RUN: PARSE COMMENTS AND PROPOSE TEXT LABEL RESOLUTIONS
#
# Applies to:
#   OTHER
#   UNKPLANTS where comments explicitly identify them
#
# Nothing is written to disk.
# ============================================================

import re
import html
import pandas as pd
import numpy as np


# ------------------------------------------------------------
# Explicit taxonomic aliases already established in this
# recovery workflow.
# ------------------------------------------------------------

TAXON_ALIASES = {
    "LADEANIA LANCEOLATA": "PSLA3",
    "AMBROSIA ACANTHICARPA": "AMAC2",

    # spelling used in comments
    "ARTEMESIA SPINESCENS": "PIDE4",

    # allow corrected spelling too
    "ARTEMISIA SPINESCENS": "PIDE4",
}


def normalize_comment(value):
    """
    Normalize formatting only.
    Do not alter biological meaning.
    """

    if value is None:
        return ""

    try:
        if pd.isna(value):
            return ""
    except (TypeError, ValueError):
        pass

    text = html.unescape(str(value))

    text = text.replace(
        "\xa0",
        " "
    )

    text = re.sub(
        r"\s+",
        " ",
        text
    )

    return text.strip()


def canonical_label(value):
    """
    Convert a parsed label to its canonical output value.
    """

    text = normalize_comment(value)

    text = re.sub(
        r"^[\s,;:]+|[\s,;:.]+$",
        "",
        text
    )

    text = re.sub(
        r"\s+",
        " ",
        text
    )

    upper = text.upper()

    return TAXON_ALIASES.get(
        upper,
        upper
    )


# ============================================================
# POINT-LIST UTILITIES
# ============================================================

def expand_point_spec(spec):
    """
    Examples:

        '6-9, 16-18,26-28,70'
            ->
        [6,7,8,9,16,17,18,26,27,28,70]

        '29,57,58,89,90,97'
            ->
        [29,57,58,89,90,97]
    """

    points = []

    for part in spec.split(","):

        part = part.strip()

        if not part:
            continue

        range_match = re.fullmatch(
            r"(\d+)\s*-\s*(\d+)",
            part
        )

        if range_match:

            start = int(
                range_match.group(1)
            )

            end = int(
                range_match.group(2)
            )

            if start <= end:
                points.extend(
                    range(
                        start,
                        end + 1
                    )
                )

            else:
                points.extend(
                    range(
                        start,
                        end - 1,
                        -1
                    )
                )

            continue

        if re.fullmatch(
            r"\d+",
            part
        ):
            points.append(
                int(part)
            )

    return points


# ============================================================
# REGEX PATTERNS
# ============================================================

# Covers:
#
# all others are SAVE4
# All others are DISP
# all other are DEPI
# all other is CADR
# all other hits are ERTR13
# aall other hits are DEPI
# all othes are HAGL
# all others areHECO26
#
DEFAULT_OTHER_RE = re.compile(
    r"(?i)^"
    r"\s*a+l+\s+"
    r"othe(?:r|rs|s)\s*"
    r"(?:hits\s*)?"
    r"(?:are|is)\s*"
    r"(.+?)"
    r"(?="
        r"\s*,\s*all\s+UNKPLANTS\b"
        r"|"
        r"\s*,?\s*except\b"
        r"|"
        r"\s*$"
    r")"
)


# Covers:
#
# Other points; SAVE4
# other points; ERTR13
# Other point; ATCO
#
OTHER_POINTS_RE = re.compile(
    r"(?i)^"
    r"\s*other\s+points?\s*;\s*"
    r"(.+?)"
    r"\s*$"
)


# Covers explicit assignments:
#
# pts 6-9,16-18,70,ACHY
#
# pt 29,57,58,89,90,97, Ambrosia acanthicarpa,
# pt 91, ACHY
#
# pts 8,9,27,69, Ladeania Lanceolata,
# pts 12, ACHY,
# pts 54, PHHA
#
POINT_ASSIGNMENT_RE = re.compile(
    r"(?i)"
    r"(?:pts?|points?)\s+"
    r"("
        r"(?:"
            r"\d+\s*"
            r"(?:-\s*\d+)?"
            r"\s*,\s*"
        r")*"
        r"\d+\s*"
        r"(?:-\s*\d+)?"
    r")"
    r"(?:\s*,\s*|\s+)"
    r"(.+?)"
    r"(?="
        r"\s*,\s*"
        r"(?:pts?|points?)\b"
        r"|"
        r"$"
    r")"
)


# Covers:
#
# except pts 21,42, are CHJU
# except pts 50,80, are CHJU
#
EXCEPTION_RE = re.compile(
    r"(?i)"
    r"except\s+"
    r"(?:pts?|points?)\s+"
    r"("
        r"(?:"
            r"\d+\s*"
            r"(?:-\s*\d+)?"
            r"\s*,\s*"
        r")*"
        r"\d+\s*"
        r"(?:-\s*\d+)?"
    r")"
    r"\s*,?\s*"
    r"(?:are|is)?\s*"
    r"([A-Za-z][A-Za-z0-9 ]+?)"
    r"\s*$"
)


# Covers:
#
# all UNKPLANTS are BAAM4
# all UNKPLANTS are ERCI6
#
UNKPLANTS_RE = re.compile(
    r"(?i)"
    r"\ball\s+UNKPLANTS\s+are\s+"
    r"([A-Za-z][A-Za-z0-9 ]+?)"
    r"(?=\s*,|$)"
)


# ============================================================
# COMMENT → RULES
# ============================================================

def parse_comment_rules(comment):

    text = normalize_comment(
        comment
    )

    rules = {
        "DefaultOTHER": None,
        "PointAssignments": {},
        "UNKPLANTS": None,
    }

    if not text:
        return rules


    # --------------------------------------------------------
    # Default OTHER assignment
    # --------------------------------------------------------

    match = DEFAULT_OTHER_RE.search(
        text
    )

    if match is not None:

        rules["DefaultOTHER"] = (
            canonical_label(
                match.group(1)
            )
        )

    else:

        match = OTHER_POINTS_RE.search(
            text
        )

        if match is not None:

            rules["DefaultOTHER"] = (
                canonical_label(
                    match.group(1)
                )
            )


    # --------------------------------------------------------
    # Explicit point-specific assignments
    # --------------------------------------------------------

    for match in (
        POINT_ASSIGNMENT_RE.finditer(
            text
        )
    ):

        points = expand_point_spec(
            match.group(1)
        )

        label = canonical_label(
            match.group(2)
        )

        for point in points:
            rules[
                "PointAssignments"
            ][point] = label


    # --------------------------------------------------------
    # "except pts ..." overrides default
    # --------------------------------------------------------

    match = EXCEPTION_RE.search(
        text
    )

    if match is not None:

        points = expand_point_spec(
            match.group(1)
        )

        label = canonical_label(
            match.group(2)
        )

        for point in points:
            rules[
                "PointAssignments"
            ][point] = label


    # --------------------------------------------------------
    # UNKPLANTS assignment
    # --------------------------------------------------------

    match = UNKPLANTS_RE.search(
        text
    )

    if match is not None:

        rules["UNKPLANTS"] = (
            canonical_label(
                match.group(1)
            )
        )


    return rules


# ============================================================
# READ TEXT-ONLY WORKBOOK
# ============================================================

xls = pd.ExcelFile(
    TEXT_XLSX
)

assert len(xls.sheet_names) >= 2

AW120_SHEET = xls.sheet_names[0]
CAM5_SHEET = xls.sheet_names[1]


resolution_records = []


for sheet_name, camera in [
    (AW120_SHEET, "AW120"),
    (CAM5_SHEET, "Cam5"),
]:

    df = pd.read_excel(
        TEXT_XLSX,
        sheet_name=sheet_name
    )

    photo_col = find_df_column(
        df,
        ["Image", "Photo", "Photos"]
    )

    comment_col = find_df_column(
        df,
        ["Comment", "Comments"]
    )

    point_cols = {
        n: find_df_column(
            df,
            [f"Point{n}"]
        )
        for n in range(1, 101)
    }


    for idx, row in df.iterrows():

        comment = row[comment_col]

        rules = parse_comment_rules(
            comment
        )

        photo_number = (
            extract_photo_number(
                row[photo_col]
            )
        )


        for point_number, point_col in (
            point_cols.items()
        ):

            value = row[point_col]

            if pd.isna(value):
                continue

            current = (
                str(value)
                .strip()
                .upper()
            )


            # =================================================
            # OTHER
            # =================================================

            if current == "OTHER":

                # Explicit point assignments have precedence.
                proposed = (
                    rules[
                        "PointAssignments"
                    ].get(
                        point_number
                    )
                )

                rule_type = None

                if proposed is not None:

                    rule_type = (
                        "point-specific"
                    )

                else:

                    proposed = (
                        rules[
                            "DefaultOTHER"
                        ]
                    )

                    if proposed is not None:

                        rule_type = (
                            "default-OTHER"
                        )


                resolution_records.append({
                    "Camera":
                        camera,

                    "Sheet":
                        sheet_name,

                    "ExcelRow":
                        int(idx) + 2,

                    "PhotoNumber":
                        photo_number,

                    "Point":
                        point_number,

                    "Original":
                        "OTHER",

                    "Proposed":
                        proposed,

                    "Rule":
                        rule_type,

                    "Comment":
                        normalize_comment(
                            comment
                        ),
                })


            # =================================================
            # UNKPLANTS
            # =================================================

            elif current == "UNKPLANTS":

                proposed = (
                    rules["UNKPLANTS"]
                )

                resolution_records.append({
                    "Camera":
                        camera,

                    "Sheet":
                        sheet_name,

                    "ExcelRow":
                        int(idx) + 2,

                    "PhotoNumber":
                        photo_number,

                    "Point":
                        point_number,

                    "Original":
                        "UNKPLANTS",

                    "Proposed":
                        proposed,

                    "Rule":
                        (
                            "UNKPLANTS-comment"
                            if proposed
                            else None
                        ),

                    "Comment":
                        normalize_comment(
                            comment
                        ),
                })


resolution_plan = pd.DataFrame(
    resolution_records
)


# ============================================================
# QA
# ============================================================

resolution_plan["Resolved"] = (
    resolution_plan[
        "Proposed"
    ].notna()
)


print("=" * 72)
print("COMMENT PARSER DRY RUN")
print("=" * 72)


summary_by_type = (
    resolution_plan
    .groupby(
        "Original"
    )
    .agg(
        Total=(
            "Point",
            "size"
        ),
        Resolved=(
            "Resolved",
            "sum"
        ),
    )
)

summary_by_type["Unresolved"] = (
    summary_by_type["Total"]
    - summary_by_type["Resolved"]
)

display(
    summary_by_type
)


print("\nProposed labels:")

display(
    resolution_plan[
        resolution_plan["Resolved"]
    ]
    .groupby(
        [
            "Original",
            "Proposed",
        ]
    )
    .size()
    .rename("Hits")
    .reset_index()
    .sort_values(
        "Hits",
        ascending=False
    )
)


unresolved = (
    resolution_plan[
        ~resolution_plan["Resolved"]
    ]
    .sort_values(
        [
            "Camera",
            "PhotoNumber",
            "Point",
        ]
    )
    .reset_index(drop=True)
)


print(
    f"\nTotal unresolved cells: "
    f"{len(unresolved):,}"
)

display(
    unresolved
)

COMMENT PARSER DRY RUN


,Total,Resolved,Unresolved
Original,,,
OTHER,4765,4753,12



Proposed labels:


,Original,Proposed,Hits
24,OTHER,SAVE4,1771
14,OTHER,ERTR13,830
13,OTHER,ERNA10,456
17,OTHER,HECO26,338
5,OTHER,ATCO,237
11,OTHER,DISP,192
3,OTHER,ARAR8,182
10,OTHER,DEPI,151
8,OTHER,CHJU,106
0,OTHER,ACHY,88



Total unresolved cells: 12


,Camera,Sheet,ExcelRow,PhotoNumber,Point,Original,Proposed,Rule,Comment,Resolved
0,AW120,AW120_Master,149,9683,2,OTHER,NaN,NaN,,False
1,Cam5,Cam5_Master,768,8213,34,OTHER,NaN,NaN,,False
2,Cam5,Cam5_Master,845,8291,64,OTHER,NaN,NaN,"pt 29,57,58,89,90,97, Ambrosia acanthicarpa, p...",False
3,Cam5,Cam5_Master,184,8645,60,OTHER,NaN,NaN,,False
4,Cam5,Cam5_Master,248,8709,75,OTHER,NaN,NaN,,False
5,Cam5,Cam5_Master,416,8880,67,OTHER,NaN,NaN,,False
6,Cam5,Cam5_Master,417,8881,74,OTHER,NaN,NaN,,False
7,Cam5,Cam5_Master,417,8881,75,OTHER,NaN,NaN,,False
8,Cam5,Cam5_Master,608,9072,4,OTHER,NaN,NaN,,False
9,Cam5,Cam5_Master,608,9072,5,OTHER,NaN,NaN,,False


In [63]:
# ============================================================
# WRITE COMMENT-RESOLVED TEXT-ONLY WORKBOOK
#
# Uses the already-audited resolution_plan.
#
# Resolved cells are replaced.
# Unresolved cells remain exactly as they are.
# Original TextOnly workbook is untouched.
# ============================================================

from pathlib import Path
import shutil
import openpyxl


TEXT_RESOLVED_XLSX = TEXT_XLSX.with_name(
    "2026_SamplePoint_Camera_Masters_"
    "RECOVERED_COMPLETE_TextOnly_OTHER_resolved.xlsx"
)


# ------------------------------------------------------------
# Copy source TextOnly workbook
# ------------------------------------------------------------

shutil.copy2(
    TEXT_XLSX,
    TEXT_RESOLVED_XLSX
)


# ------------------------------------------------------------
# Open copy
# ------------------------------------------------------------

wb = openpyxl.load_workbook(
    TEXT_RESOLVED_XLSX
)


# Header maps by sheet
header_maps = {}

for sheet_name in [
    wb.sheetnames[0],   # AW120
    wb.sheetnames[1],   # Cam5
]:

    ws = wb[sheet_name]

    header_maps[sheet_name] = {
        str(cell.value).strip(): cell.column
        for cell in ws[1]
        if cell.value is not None
    }


# ------------------------------------------------------------
# Apply only audited/resolved changes
# ------------------------------------------------------------

resolved_plan = (
    resolution_plan[
        resolution_plan["Resolved"]
    ]
    .copy()
)


changes_written = 0


for _, rec in resolved_plan.iterrows():

    sheet_name = rec["Sheet"]
    excel_row = int(
        rec["ExcelRow"]
    )
    point_number = int(
        rec["Point"]
    )

    original_expected = str(
        rec["Original"]
    ).strip().upper()

    proposed = str(
        rec["Proposed"]
    ).strip()

    ws = wb[sheet_name]

    point_col = header_maps[
        sheet_name
    ][
        f"Point{point_number}"
    ]

    cell = ws.cell(
        row=excel_row,
        column=point_col
    )

    current = (
        ""
        if cell.value is None
        else str(cell.value).strip().upper()
    )

    # Safety gate: never overwrite something unexpected.
    if current != original_expected:

        raise ValueError(
            f"Unexpected cell contents before replacement:\n"
            f"Sheet={sheet_name}\n"
            f"Row={excel_row}\n"
            f"Point={point_number}\n"
            f"Expected={original_expected!r}\n"
            f"Found={current!r}"
        )

    cell.value = proposed

    changes_written += 1


wb.save(
    TEXT_RESOLVED_XLSX
)

wb.close()


print("=" * 72)
print("COMMENT RESOLUTION WRITTEN")
print("=" * 72)

print(
    f"Resolved cells written: "
    f"{changes_written:,}"
)

print(
    f"\nOutput:\n"
    f"{TEXT_RESOLVED_XLSX}"
)

COMMENT RESOLUTION WRITTEN
Resolved cells written: 4,753

Output:
D:\My Drive\BOP_OCTC_2025\Photos\2026\Cropped\XLS_OutputFiles\2026_SamplePoint_Camera_Masters_RECOVERED_COMPLETE_TextOnly_OTHER_resolved.xlsx


In [62]:
# ============================================================
# FINAL QA:
# COUNT REMAINING OTHER / UNKPLANTS
# ============================================================

remaining_records = []


xls = pd.ExcelFile(
    TEXT_RESOLVED_XLSX
)


for sheet_name, camera in [
    (xls.sheet_names[0], "AW120"),
    (xls.sheet_names[1], "Cam5"),
]:

    df = pd.read_excel(
        TEXT_RESOLVED_XLSX,
        sheet_name=sheet_name
    )

    photo_col = find_df_column(
        df,
        ["Image", "Photo", "Photos"]
    )

    comment_col = find_df_column(
        df,
        ["Comment", "Comments"]
    )

    point_cols = {
        n: find_df_column(
            df,
            [f"Point{n}"]
        )
        for n in range(1, 101)
    }


    for idx, row in df.iterrows():

        for point_number, point_col in (
            point_cols.items()
        ):

            value = row[point_col]

            if pd.isna(value):
                continue

            label = str(
                value
            ).strip().upper()

            if label not in {
                "OTHER",
                "UNKPLANTS",
            }:
                continue

            remaining_records.append({
                "Camera":
                    camera,

                "Sheet":
                    sheet_name,

                "ExcelRow":
                    int(idx) + 2,

                "PhotoNumber":
                    extract_photo_number(
                        row[photo_col]
                    ),

                "Point":
                    point_number,

                "Label":
                    label,

                "Comment":
                    (
                        ""
                        if pd.isna(row[comment_col])
                        else str(
                            row[comment_col]
                        ).strip()
                    ),
            })


remaining_unknowns = pd.DataFrame(
    remaining_records
)


print("=" * 72)
print("FINAL UNKNOWN-LABEL QA")
print("=" * 72)

print(
    f"Remaining OTHER / UNKPLANTS: "
    f"{len(remaining_unknowns):,}"
)


if len(remaining_unknowns):

    display(
        remaining_unknowns.sort_values(
            [
                "Camera",
                "PhotoNumber",
                "Point",
            ]
        )
        .reset_index(drop=True)
    )

FINAL UNKNOWN-LABEL QA
Remaining OTHER / UNKPLANTS: 12


,Camera,Sheet,ExcelRow,PhotoNumber,Point,Label,Comment
0,AW120,AW120_Master,149,9683,2,OTHER,
1,Cam5,Cam5_Master,768,8213,34,OTHER,
2,Cam5,Cam5_Master,845,8291,64,OTHER,"pt 29,57,58,89,90,97, Ambrosia acanthicarpa, p..."
3,Cam5,Cam5_Master,184,8645,60,OTHER,
4,Cam5,Cam5_Master,248,8709,75,OTHER,
5,Cam5,Cam5_Master,416,8880,67,OTHER,
6,Cam5,Cam5_Master,417,8881,74,OTHER,
7,Cam5,Cam5_Master,417,8881,75,OTHER,
8,Cam5,Cam5_Master,608,9072,4,OTHER,
9,Cam5,Cam5_Master,608,9072,5,OTHER,


In [70]:
# =============================================================================
# SYNCHRONIZE 2026_SamplePoint_PointRaw.xlsx TO 2026 Master Survey.xlsx
#
# IMPORTANT:
#   PointRaw is the authoritative classification source because its
#   Point1-Point100 values include the 12 manually resolved OTHER cells.
#
# This cell changes ONLY:
#   - stale photo identities
#   - stale duplicate rows
#   - Import_Log row counts
#
# It does NOT alter Point1-Point100 for any retained classification.
# =============================================================================

from pathlib import Path
import re
import os
import shutil
import hashlib

import numpy as np
import pandas as pd
import openpyxl


# =============================================================================
# PATHS
# =============================================================================

PROJECT_ROOT = Path(
    r"D:\My Drive\BOP_OCTC_2025"
)

XLS_DIR = (
    PROJECT_ROOT
    / "Photos"
    / "2026"
    / "Cropped"
    / "XLS_OutputFiles"
)

POINT_RAW = (
    XLS_DIR
    / "2026_SamplePoint_PointRaw.xlsx"
)

MASTER_SURVEY = (
    PROJECT_ROOT
    / "2026 Master Survey.xlsx"
)

BACKUP = (
    XLS_DIR
    / "2026_SamplePoint_PointRaw_PRE_SURVEY_SYNC.xlsx"
)

TEMP = (
    XLS_DIR
    / "2026_SamplePoint_PointRaw_SURVEY_SYNC_TEMP.xlsx"
)

SURVEY_SHEET = "Sheet1"

PHOTO_COLUMNS = [
    "Center Photoplot",
    "North Photoplot",
    "East Photoplot",
    "South Photoplot",
    "West Photoplot",
]

POINT_COLUMNS = [
    f"Point{i}"
    for i in range(1, 101)
]


# =============================================================================
# HELPERS
# =============================================================================

def clean_text(value):

    if value is None:
        return None

    try:
        if pd.isna(value):
            return None
    except (TypeError, ValueError):
        pass

    text = str(value).strip()

    if text == "" or text.lower() == "nan":
        return None

    return text


def normalize_database(value):

    text = clean_text(value)

    if text is None:
        return None

    return re.sub(
        r"\.xlsx?$",
        "",
        text.lower(),
    )


def normalize_camera(value):

    text = clean_text(value)

    if text is None:
        return None

    low = text.lower()

    if "aw120" in low:
        return "AW120"

    if (
        "cam5" in low
        or "cam 5" in low
        or "camera5" in low
        or "camera 5" in low
    ):
        return "Cam5"

    return None


def extract_photo_number(value):

    if value is None:
        return None

    if isinstance(
        value,
        (int, np.integer),
    ):
        return int(value)

    if isinstance(
        value,
        (float, np.floating),
    ):
        if np.isnan(value):
            return None

        if float(value).is_integer():
            return int(value)

    text = str(value).strip()

    match = re.search(
        r"DSCN0*(\d+)",
        text,
        flags=re.IGNORECASE,
    )

    if match:
        return int(match.group(1))

    match = re.fullmatch(
        r"0*(\d+)(?:\.0+)?",
        text,
    )

    if match:
        return int(match.group(1))

    return None


def replace_photo_number(
    value,
    new_number,
):

    text = str(value)

    match = re.search(
        r"(?i)DSCN0*\d+",
        text,
    )

    if match:

        return (
            text[:match.start()]
            + f"DSCN{int(new_number)}"
            + text[match.end():]
        )

    if re.fullmatch(
        r"\s*\d+(?:\.0+)?\s*",
        text,
    ):
        return int(new_number)

    raise ValueError(
        f"Cannot safely replace photo number in {value!r}"
    )


def point_signature(row):

    values = []

    for col in POINT_COLUMNS:

        value = row[col]

        if pd.isna(value):
            value = "<NA>"
        else:
            value = str(value)

        values.append(value)

    payload = "\x1f".join(
        values
    ).encode(
        "utf-8"
    )

    return hashlib.sha256(
        payload
    ).hexdigest()


# =============================================================================
# READ CURRENT POINT RAW
# =============================================================================

for path in [
    POINT_RAW,
    MASTER_SURVEY,
]:
    if not path.exists():
        raise FileNotFoundError(path)


xls = pd.ExcelFile(
    POINT_RAW
)

AW120_SHEET = xls.sheet_names[0]
CAM5_SHEET = xls.sheet_names[1]

if "Import_Log" not in xls.sheet_names:
    raise ValueError(
        "PointRaw is missing Import_Log."
    )


import_log = pd.read_excel(
    POINT_RAW,
    sheet_name="Import_Log",
).copy()

import_log["DatabaseNorm"] = (
    import_log["Database"]
    .map(normalize_database)
)


# =============================================================================
# RECONSTRUCT SOURCE DATABASE FOR EACH POINT RAW ROW
# =============================================================================

frames = []


for (
    sheet_name,
    camera,
    count_col,
) in [

    (
        AW120_SHEET,
        "AW120",
        "RowsToAW120",
    ),

    (
        CAM5_SHEET,
        "Cam5",
        "RowsToCam5",
    ),
]:

    df = pd.read_excel(
        POINT_RAW,
        sheet_name=sheet_name,
    ).copy()


    missing_points = (
        set(POINT_COLUMNS)
        - set(df.columns)
    )

    if missing_points:

        raise ValueError(
            f"{sheet_name} is missing Point columns."
        )


    source_database = []


    for _, rec in import_log.iterrows():

        n = rec[count_col]

        n = (
            0
            if pd.isna(n)
            else int(n)
        )

        source_database.extend(
            [rec["DatabaseNorm"]] * n
        )


    if len(source_database) != len(df):

        raise ValueError(
            f"{sheet_name}: Import_Log reconstructs "
            f"{len(source_database):,} rows but the sheet "
            f"contains {len(df):,}."
        )


    df["SourceDatabase"] = (
        source_database
    )

    df["Camera"] = camera

    df["ExcelRow"] = (
        np.arange(
            2,
            len(df) + 2,
        )
    )

    df["PhotoNumber"] = (
        df["image"]
        .map(extract_photo_number)
        .astype(int)
    )

    df["SourceSequence"] = (
        df.groupby(
            [
                "SourceDatabase",
                "Camera",
            ],
            sort=False,
        )
        .cumcount()
    )

    df["PointSignature"] = (
        df.apply(
            point_signature,
            axis=1,
        )
    )


    frames.append(df)


point_raw = pd.concat(
    frames,
    ignore_index=True,
)


if len(point_raw) != 2280:

    raise ValueError(
        f"Expected 2,280 PointRaw rows; "
        f"found {len(point_raw):,}."
    )


# =============================================================================
# READ + FLATTEN CORRECTED MASTER SURVEY
# =============================================================================

survey = pd.read_excel(
    MASTER_SURVEY,
    sheet_name=SURVEY_SHEET,
).copy()

survey["_SurveyRow"] = (
    np.arange(len(survey))
)

survey["DatabaseNorm"] = (
    survey[
        "Samplepoint Database Name"
    ]
    .map(normalize_database)
)

survey["Camera"] = (
    survey[
        "Camera Number"
    ]
    .map(normalize_camera)
)


survey_records = []


for _, rec in survey.iterrows():

    for (
        position_order,
        position,
    ) in enumerate(
        PHOTO_COLUMNS
    ):

        photo = extract_photo_number(
            rec[position]
        )

        if photo is None:
            continue


        survey_records.append({

            "DatabaseNorm":
                rec["DatabaseNorm"],

            "Camera":
                rec["Camera"],

            "PhotoNumber":
                int(photo),

            "_SurveyRow":
                int(rec["_SurveyRow"]),

            "PhotoPositionOrder":
                position_order,
        })


survey_photos = (
    pd.DataFrame(
        survey_records
    )
    .sort_values(
        [
            "_SurveyRow",
            "PhotoPositionOrder",
        ]
    )
    .reset_index(drop=True)
)


survey_photos["SurveySequence"] = (
    survey_photos.groupby(
        [
            "DatabaseNorm",
            "Camera",
        ],
        sort=False,
        dropna=False,
    )
    .cumcount()
)


# =============================================================================
# CLASSIFY EACH POINT RAW ROW
# =============================================================================

survey_exact = set(
    zip(
        survey_photos["DatabaseNorm"],
        survey_photos["Camera"],
        survey_photos["PhotoNumber"],
    )
)

survey_camera_photo = set(
    zip(
        survey_photos["Camera"],
        survey_photos["PhotoNumber"],
    )
)


point_raw["Action"] = pd.NA
point_raw["NewPhotoNumber"] = (
    point_raw["PhotoNumber"]
)


# -----------------------------------------------------------------------------
# 1. Exact survey matches are already correct
# -----------------------------------------------------------------------------

exact_mask = point_raw.apply(
    lambda r: (
        r["SourceDatabase"],
        r["Camera"],
        int(r["PhotoNumber"]),
    ) in survey_exact,
    axis=1,
)

point_raw.loc[
    exact_mask,
    "Action",
] = "KEEP"


# -----------------------------------------------------------------------------
# 2. Known legitimate survey exceptions remain unchanged
# -----------------------------------------------------------------------------

known_exception_mask = (
    point_raw["Action"].isna()
    &
    point_raw["Camera"].eq("AW120")
    &
    point_raw["PhotoNumber"].isin(
        [
            9295,
            9296,
            9297,
            9298,
            9299,
            9915,
        ]
    )
)

point_raw.loc[
    known_exception_mask,
    "Action",
] = "KEEP_EXCEPTION"


# =============================================================================
# 3. FIX cam5_database_15 BY VERIFIED SURVEY SEQUENCE
# =============================================================================

cam15_source = (
    point_raw[
        point_raw[
            "SourceDatabase"
        ].eq(
            "cam5_database_15"
        )
        &
        point_raw[
            "Camera"
        ].eq(
            "Cam5"
        )
    ]
    .sort_values(
        "SourceSequence"
    )
    .copy()
)


cam15_survey = (
    survey_photos[
        survey_photos[
            "DatabaseNorm"
        ].eq(
            "cam5_database_15"
        )
        &
        survey_photos[
            "Camera"
        ].eq(
            "Cam5"
        )
    ]
    .sort_values(
        "SurveySequence"
    )
    .copy()
)


if (
    len(cam15_source) != 40
    or len(cam15_survey) != 40
):

    raise ValueError(
        "Expected 40 rows in both PointRaw and survey "
        "for cam5_database_15."
    )


cam15_compare = (
    cam15_source[
        [
            "SourceSequence",
            "ExcelRow",
            "PhotoNumber",
        ]
    ]
    .merge(
        cam15_survey[
            [
                "SurveySequence",
                "PhotoNumber",
            ]
        ],
        left_on="SourceSequence",
        right_on="SurveySequence",
        suffixes=(
            "_Raw",
            "_Survey",
        ),
        validate="one_to_one",
    )
)


cam15_compare["Matches"] = (
    cam15_compare[
        "PhotoNumber_Raw"
    ]
    .eq(
        cam15_compare[
            "PhotoNumber_Survey"
        ]
    )
)


# The current known state is 35 already correct + 5 stale IDs.
if (
    cam15_compare[
        "Matches"
    ].sum()
    != 35
):

    display(
        cam15_compare
    )

    raise ValueError(
        "cam5_database_15 no longer has the expected "
        "35 matching + 5 stale sequence pattern."
    )


cam15_fixes = (
    cam15_compare[
        ~cam15_compare[
            "Matches"
        ]
    ]
    .copy()
)


if set(
    cam15_fixes[
        "PhotoNumber_Raw"
    ].astype(int)
) != {
    9161,
    9162,
    9163,
    9164,
    9165,
}:

    raise ValueError(
        "Unexpected cam5_database_15 stale photo set."
    )


for _, fix in (
    cam15_fixes.iterrows()
):

    mask = (
        point_raw[
            "SourceDatabase"
        ].eq(
            "cam5_database_15"
        )
        &
        point_raw[
            "Camera"
        ].eq(
            "Cam5"
        )
        &
        point_raw[
            "ExcelRow"
        ].eq(
            int(fix["ExcelRow"])
        )
    )


    point_raw.loc[
        mask,
        "Action",
    ] = "RENUMBER"

    point_raw.loc[
        mask,
        "NewPhotoNumber",
    ] = int(
        fix[
            "PhotoNumber_Survey"
        ]
    )


# =============================================================================
# 4. ANY REMAINING ROW WHOSE PHOTO BELONGS TO ANOTHER SURVEY DATABASE
#    IS A STALE SOURCE DUPLICATE
# =============================================================================

remaining_mask = (
    point_raw["Action"].isna()
)


for idx in point_raw.index[
    remaining_mask
]:

    camera = (
        point_raw.at[
            idx,
            "Camera",
        ]
    )

    photo = int(
        point_raw.at[
            idx,
            "PhotoNumber",
        ]
    )


    if (
        camera,
        photo,
    ) in survey_camera_photo:

        point_raw.at[
            idx,
            "Action",
        ] = "DELETE_STALE"

    else:

        point_raw.at[
            idx,
            "Action",
        ] = "UNRESOLVED"


# =============================================================================
# HARD STOP ON ANY UNEXPLAINED ROW
# =============================================================================

unresolved = (
    point_raw[
        point_raw[
            "Action"
        ].eq(
            "UNRESOLVED"
        )
    ]
)


if len(unresolved):

    display(
        unresolved[
            [
                "SourceDatabase",
                "Camera",
                "PhotoNumber",
                "ExcelRow",
            ]
        ]
    )

    raise ValueError(
        "Unresolved PointRaw identities remain. "
        "Nothing has been written."
    )


# Current expected synchronization:
#   6 stale rows removed
#   5 Cam5 rows renumbered

to_delete = (
    point_raw[
        point_raw[
            "Action"
        ].eq(
            "DELETE_STALE"
        )
    ]
    .copy()
)

to_renumber = (
    point_raw[
        point_raw[
            "Action"
        ].eq(
            "RENUMBER"
        )
    ]
    .copy()
)


if len(to_delete) != 6:

    display(to_delete)

    raise ValueError(
        f"Expected 6 stale rows; found {len(to_delete)}."
    )


if len(to_renumber) != 5:

    display(to_renumber)

    raise ValueError(
        f"Expected 5 photo renumberings; "
        f"found {len(to_renumber)}."
    )


# =============================================================================
# EXPECTED POINT SIGNATURES AFTER SYNCHRONIZATION
#
# This protects the manually resolved Point1-Point100 classifications.
# =============================================================================

expected_signatures = {}


for (
    database,
    camera,
), group in (
    point_raw[
        ~point_raw[
            "Action"
        ].eq(
            "DELETE_STALE"
        )
    ]
    .groupby(
        [
            "SourceDatabase",
            "Camera",
        ],
        sort=False,
    )
):

    expected_signatures[
        (
            database,
            camera,
        )
    ] = (
        group.sort_values(
            "SourceSequence"
        )[
            "PointSignature"
        ]
        .tolist()
    )


# =============================================================================
# MAKE BACKUP + TEMP WORKBOOK
# =============================================================================

if not BACKUP.exists():

    shutil.copy2(
        POINT_RAW,
        BACKUP,
    )

    print(
        "Backup created:"
    )

    print(
        BACKUP
    )

else:

    print(
        "Backup already exists:"
    )

    print(
        BACKUP
    )


if TEMP.exists():
    TEMP.unlink()


shutil.copy2(
    POINT_RAW,
    TEMP,
)


# =============================================================================
# APPLY CHANGES TO TEMP WORKBOOK
# =============================================================================

wb = openpyxl.load_workbook(
    TEMP
)


# -----------------------------------------------------------------------------
# Renumber the five Cam5 image identities
# -----------------------------------------------------------------------------

cam_ws = wb[
    CAM5_SHEET
]

cam_headers = {
    str(cell.value).strip():
        cell.column

    for cell in cam_ws[1]

    if cell.value is not None
}

image_col = (
    cam_headers["image"]
)


for _, rec in (
    to_renumber.iterrows()
):

    excel_row = int(
        rec["ExcelRow"]
    )

    old_photo = int(
        rec["PhotoNumber"]
    )

    new_photo = int(
        rec["NewPhotoNumber"]
    )


    cell = cam_ws.cell(
        row=excel_row,
        column=image_col,
    )


    if (
        extract_photo_number(
            cell.value
        )
        != old_photo
    ):

        raise ValueError(
            f"Unexpected Cam5 value at row {excel_row}."
        )


    cell.value = replace_photo_number(
        cell.value,
        new_photo,
    )


# -----------------------------------------------------------------------------
# Delete stale rows, bottom-to-top by sheet
# -----------------------------------------------------------------------------

for (
    sheet_name,
    camera,
) in [

    (
        AW120_SHEET,
        "AW120",
    ),

    (
        CAM5_SHEET,
        "Cam5",
    ),
]:

    ws = wb[
        sheet_name
    ]

    delete_rows = (
        to_delete.loc[
            to_delete[
                "Camera"
            ].eq(
                camera
            ),
            "ExcelRow",
        ]
        .astype(int)
        .sort_values(
            ascending=False
        )
        .tolist()
    )


    for excel_row in (
        delete_rows
    ):

        ws.delete_rows(
            excel_row,
            1,
        )


# =============================================================================
# UPDATE Import_Log COUNTS FOR THE SIX DELETED ROWS
# =============================================================================

log_ws = wb[
    "Import_Log"
]

log_headers = {
    str(cell.value).strip():
        cell.column

    for cell in log_ws[1]

    if cell.value is not None
}


delete_counts = (
    to_delete
    .groupby(
        [
            "SourceDatabase",
            "Camera",
        ]
    )
    .size()
    .to_dict()
)


for row in range(
    2,
    log_ws.max_row + 1,
):

    db = normalize_database(
        log_ws.cell(
            row=row,
            column=log_headers[
                "Database"
            ],
        ).value
    )

    if db is None:
        continue


    for (
        camera,
        count_column,
    ) in [

        (
            "AW120",
            "RowsToAW120",
        ),

        (
            "Cam5",
            "RowsToCam5",
        ),
    ]:

        n_delete = (
            delete_counts.get(
                (
                    db,
                    camera,
                ),
                0,
            )
        )

        if n_delete == 0:
            continue


        cell = log_ws.cell(
            row=row,
            column=log_headers[
                count_column
            ],
        )


        cell.value = (
            int(cell.value)
            - int(n_delete)
        )


# =============================================================================
# SAVE TEMP
# =============================================================================

wb.save(
    TEMP
)

wb.close()


# =============================================================================
# VERIFY FINISHED TEMP WORKBOOK
# =============================================================================

final_xls = pd.ExcelFile(
    TEMP
)

final_import_log = pd.read_excel(
    TEMP,
    sheet_name="Import_Log",
).copy()

final_import_log[
    "DatabaseNorm"
] = (
    final_import_log[
        "Database"
    ]
    .map(
        normalize_database
    )
)


final_frames = []


for (
    sheet_name,
    camera,
    count_col,
) in [

    (
        final_xls.sheet_names[0],
        "AW120",
        "RowsToAW120",
    ),

    (
        final_xls.sheet_names[1],
        "Cam5",
        "RowsToCam5",
    ),
]:

    df = pd.read_excel(
        TEMP,
        sheet_name=sheet_name,
    ).copy()


    source_database = []


    for _, rec in (
        final_import_log.iterrows()
    ):

        n = rec[
            count_col
        ]

        n = (
            0
            if pd.isna(n)
            else int(n)
        )

        source_database.extend(
            [
                rec[
                    "DatabaseNorm"
                ]
            ] * n
        )


    if len(source_database) != len(df):

        raise ValueError(
            "Updated Import_Log does not match "
            f"{sheet_name}."
        )


    df[
        "SourceDatabase"
    ] = (
        source_database
    )

    df[
        "Camera"
    ] = camera

    df[
        "PhotoNumber"
    ] = (
        df["image"]
        .map(
            extract_photo_number
        )
        .astype(int)
    )

    df[
        "PointSignature"
    ] = (
        df.apply(
            point_signature,
            axis=1,
        )
    )


    final_frames.append(
        df
    )


final = pd.concat(
    final_frames,
    ignore_index=True,
)


# -----------------------------------------------------------------------------
# 1. Exact row count
# -----------------------------------------------------------------------------

if len(final) != 2274:

    raise ValueError(
        f"Expected 2,274 final rows; "
        f"found {len(final):,}."
    )


# -----------------------------------------------------------------------------
# 2. No duplicate physical photo identities
# -----------------------------------------------------------------------------

duplicates = (
    final[
        final.duplicated(
            [
                "Camera",
                "PhotoNumber",
            ],
            keep=False,
        )
    ]
)


if len(duplicates):

    display(
        duplicates
    )

    raise ValueError(
        "Duplicate camera/photo identities remain."
    )


# -----------------------------------------------------------------------------
# 3. Every retained row still has all 100 classified points
# -----------------------------------------------------------------------------

n_points = (
    final[
        POINT_COLUMNS
    ]
    .notna()
    .sum(axis=1)
)


if not n_points.eq(100).all():

    raise ValueError(
        "A retained row lost Point1-Point100 data."
    )


# -----------------------------------------------------------------------------
# 4. No manual OTHER resolutions were lost
# -----------------------------------------------------------------------------

point_text = (
    final[
        POINT_COLUMNS
    ]
    .astype(str)
    .apply(
        lambda x:
        x.str.strip().str.upper()
    )
)


remaining_other = int(
    point_text
    .eq("OTHER")
    .sum()
    .sum()
)

remaining_unknown = int(
    point_text
    .isin(
        [
            "UNKPLANT",
            "UNKPLANTS",
        ]
    )
    .sum()
    .sum()
)


if (
    remaining_other
    or remaining_unknown
):

    raise ValueError(
        "An unresolved OTHER/UNKPLANT appeared "
        "after synchronization."
    )


# -----------------------------------------------------------------------------
# 5. Point1-Point100 signatures must be EXACTLY preserved
#    within every retained source database.
# -----------------------------------------------------------------------------

for (
    database,
    camera,
), group in (
    final.groupby(
        [
            "SourceDatabase",
            "Camera",
        ],
        sort=False,
    )
):

    observed = (
        group[
            "PointSignature"
        ]
        .tolist()
    )

    expected = (
        expected_signatures.get(
            (
                database,
                camera,
            )
        )
    )


    if expected is None:

        raise ValueError(
            f"Unexpected final database block: "
            f"{database} / {camera}"
        )


    if observed != expected:

        raise ValueError(
            "Point1-Point100 values changed for "
            f"{database} / {camera}."
        )


# -----------------------------------------------------------------------------
# 6. Known retained exceptions still exist
# -----------------------------------------------------------------------------

final_keys = set(
    zip(
        final[
            "Camera"
        ],
        final[
            "PhotoNumber"
        ],
    )
)


required_exceptions = {
    ("AW120", 9295),
    ("AW120", 9296),
    ("AW120", 9297),
    ("AW120", 9298),
    ("AW120", 9299),
    ("AW120", 9915),
}


if not (
    required_exceptions
    <= final_keys
):

    raise ValueError(
        "One or more known retained survey "
        "exceptions disappeared."
    )


# =============================================================================
# TEMP PASSED ALL QA — REPLACE POINT RAW
# =============================================================================

os.replace(
    TEMP,
    POINT_RAW,
)


# =============================================================================
# REPORT
# =============================================================================

print("\n" + "=" * 78)
print("POINT RAW SYNCHRONIZED TO 2026 MASTER SURVEY")
print("=" * 78)

print(
    f"Starting PointRaw rows:          2,280"
)

print(
    f"Stale survey rows removed:           {len(to_delete):,}"
)

print(
    f"Photo identities corrected:          {len(to_renumber):,}"
)

print(
    f"Final PointRaw rows:              {len(final):,}"
)

print(
    f"Duplicate camera/photo IDs:          {len(duplicates):,}"
)

print(
    f"Rows retaining all 100 points:    {n_points.eq(100).sum():,}"
)

print(
    f"Remaining OTHER:                     {remaining_other:,}"
)

print(
    f"Remaining UNKPLANT(S):                {remaining_unknown:,}"
)

print(
    "\nPASS: Point1-Point100 signatures are unchanged "
    "for every retained row."
)

print(
    "\nPointRaw:"
)

print(
    POINT_RAW
)

print(
    "\nBackup:"
)

print(
    BACKUP
)

ValueError: Expected 40 rows in both PointRaw and survey for cam5_database_15.